# 레이더 시계열 데이터 기반 이동 표적 식별하기

본 임무의 목적은 **짧은 시간 동안 수집된 레이더 시계열 데이터를 분석하여 이동 표적의 클래스를 식별하는 분류기**를 만드는 것입니다.

가상의 기지 주변 도로와 진입로에는 일반 차량, 대형 차량, 이륜 이동수단, 보행자 등 다양한 이동 표적이 존재합니다. 레이더 센서는 하나의 이동 표적을 관측하는 동안 거리, 방위각, 반사 강도(RCS), 속도, 위치 등의 정보를 여러 시점에 걸쳐 수집합니다.

본 과제에서는 약 **500ms 동안 관측된 하나의 이동 표적**을 하나의 `sample_id`로 정의합니다. 동일한 `sample_id`에는 시간에 따라 수집된 여러 개의 레이더 detection 레코드가 포함될 수 있습니다.

주어진 훈련 데이터를 이용하여 분류 모델을 학습한 뒤, 검증 데이터의 각 `sample_id`가 아래 5개 클래스 중 어디에 해당하는지 예측해야 합니다.

| 레이블 | 설명 |
| --- | --- |
| `LIGHT_VEHICLE` | 일반 승용차급 차량 |
| `HEAVY_VEHICLE` | 대형 차량 |
| `TWO_WHEELER` | 자전거 또는 오토바이류 |
| `PERSONNEL` | 단독 보행자 |
| `PERSONNEL_GROUP` | 여러 명의 보행자 |

평가는 검증 데이터의 각 `sample_id`에 대해 예측한 이동 표적 클래스가 실제 정답과 일치하는지를 기준으로 **Accuracy(정확도)** 를 계산하여 진행합니다.

## 훈련 데이터

`train.csv`에는 총 **1,000개의 이동 표적 sample**이 제공됩니다. 5개 클래스별로 200개의 sample이 포함되어 있습니다.

하나의 sample은 여러 개의 레이더 detection 레코드로 구성되므로, `train.csv`에서 하나의 `sample_id`가 여러 행에 걸쳐 나타날 수 있습니다.

| 필드 | 타입 | 설명 |
| --- | --- | --- |
| `sample_id` | 문자열 | 하나의 이동 표적 관측 구간을 식별하는 ID |
| `t_ms` | 실수 | 해당 sample의 첫 관측 시점으로부터 경과한 시간(ms) |
| `sensor_id` | 숫자 | detection을 수집한 레이더 센서 ID |
| `range_m` | 실수 | detection을 관측한 레이더 센서로부터 표적까지의 거리(m) |
| `azimuth_rad` | 실수 | detection을 관측한 레이더 센서의 로컬 좌표계를 기준으로 측정한 방위각(rad) |
| `rcs_dbsm` | 실수 | Radar Cross Section(dBsm). 레이더 반사 강도와 관련된 값 |
| `radial_velocity_mps` | 실수 | 레이더가 측정한 방사 방향 속도(m/s) |
| `radial_velocity_comp_mps` | 실수 | 센서 이동 영향을 보정한 방사 방향 속도(m/s) |
| `x_m` | 실수 | 센서의 장착 위치와 방향을 반영하여 공통 기준 좌표계로 변환한 detection의 x 위치((m) |
| `y_m` | 실수 | 센서의 장착 위치와 방향을 반영하여 공통 기준 좌표계로 변환한 detection의 y 위치((m) |
| `label` | 문자열 | 예측 대상이 되는 이동 표적 클래스 |

range_m 과 azimuth_rad 는 detection을 수집한 개별 레이더 센서의 로컬 좌표계를 기준으로 측정된 값입니다. 반면 X_m 과 y_m는 각 센서의 장착 위치와 방향을 반 영하여 공통 기준 좌표계로 변환한 위치입니다. 따라서 일반적으로 range_m은 Sqrt(X_m2 + y_m2) 와 직접 일치하지 않을 수 있습니다.

예를 들어 동일한 `sample_id`에 다음과 같이 여러 시점의 detection이 존재할 수 있습니다.

| sample_id | t_ms | sensor_id | range_m | azimuth_rad | rcs_dbsm | radial_velocity_mps | x_m | y_m | label |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| TR_000001 | 0.0 | 1 | ... | ... | ... | ... | ... | ... | LIGHT_VEHICLE |
| TR_000001 | 63.2 | 1 | ... | ... | ... | ... | ... | ... | LIGHT_VEHICLE |
| TR_000001 | 127.5 | 2 | ... | ... | ... | ... | ... | ... | LIGHT_VEHICLE |
| TR_000002 | 0.0 | 3 | ... | ... | ... | ... | ... | ... | PERSONNEL |

위 값은 **파일 구조를 설명하기 위한 예시**이며 실제 데이터 값은 `train.csv`를 사용해야 합니다.

행 하나를 독립적으로 분류하는 것이 아니라, **같은 `sample_id`에 속한 여러 시점의 레이더 정보를 함께 활용하여 하나의 표적 클래스를 예측하는 것**이 핵심입니다.

이 데이터셋을 이용하여 이동 표적 분류 모델을 학습합니다.

## 검증 데이터

`test.csv`에는 총 **300개의 이동 표적 sample**이 포함되어 있습니다.

훈련 데이터와 동일한 레이더 필드가 제공되지만 정답인 `label` 필드는 포함되지 않습니다.

| 필드 | 타입 | 설명 |
| --- | --- | --- |
| `sample_id` | 문자열 | 분류해야 할 이동 표적 sample ID |
| `t_ms` | 실수 | 해당 sample의 첫 관측 시점으로부터 경과한 시간(ms) |
| `sensor_id` | 숫자 | detection을 수집한 레이더 센서 ID |
| `range_m` | 실수 | detection을 관측한 레이더 센서로부터 표적까지의 거리(m) |
| `azimuth_rad` | 실수 | detection을 관측한 레이더 센서의 로컬 좌표계를 기준으로 측정한 방위각(rad) |
| `rcs_dbsm` | 실수 | Radar Cross Section(dBsm) |
| `radial_velocity_mps` | 실수 | 레이더가 측정한 방사 방향 속도(m/s) |
| `radial_velocity_comp_mps` | 실수 | 센서 이동 영향을 보정한 방사 방향 속도(m/s) |
| `x_m` | 실수 | 센서의 장착 위치와 방향을 반영하여 공통 기준 좌표계로 변환한 detection의 x 위치(m) |
| `y_m` | 실수 | 센서의 장착 위치와 방향을 반영하여 공통 기준 좌표계로 변환한 detection의 y 위치(m) |

훈련 데이터로 만든 모델을 이용하여 `test.csv`의 각 `sample_id`에 대해 하나의 이동 표적 클래스를 예측하고, 아래에서 설명하는 제출 파일 형식에 맞춰 결과를 저장하면 됩니다.

## 제출 파일 포맷

`test.csv`의 각 `sample_id`에 대해 하나의 예측 결과를 생성하고 아래와 같은 형태의 CSV 파일로 저장합니다. 파일 이름은 `submission.csv`로 지정합니다.

각 `sample_id`는 결과 파일에 정확히 한 번만 나타나야 하며, 결과는 sample_id를 기준으로 오름차순 정렬합니다.

| sample_id | label |
| --- | --- |
| VA_000001 | LIGHT_VEHICLE |
| VA_000002 | PERSONNEL |
| VA_000003 | TWO_WHEELER |
| VA_000004 | HEAVY_VEHICLE |
| VA_000005 | PERSONNEL_GROUP |

위 결과는 제출 포맷을 보여주기 위한 예시이며 실제 정답과는 관계가 없습니다.

- `sample_id`: `test.csv`에 주어진 ID 값을 그대로 사용합니다.
- `label`: 다음 5개 값 중 하나를 사용합니다.
  - `LIGHT_VEHICLE`
  - `HEAVY_VEHICLE`
  - `TWO_WHEELER`
  - `PERSONNEL`
  - `PERSONNEL_GROUP`
- `test.csv`에 존재하는 모든 `sample_id`에 대해 결과가 있어야 합니다.
- 동일한 `sample_id`가 중복으로 존재하면 안 됩니다.



## 평가 기준

검증 데이터의 각 `sample_id`에 대해 예측 결과와 실제 정답이 일치하는지를 비교하여 **Accuracy(정확도)** 를 계산합니다.

* correct = 이동 표적 클래스를 정확하게 예측한 sample 수
* total = 전체 validation sample 수
* accuracy = correct / total

예를 들어 300개의 validation sample 중 225개를 정확하게 분류했다면:

`Accuracy = 225 / 300 = 0.75`

따라서 점수는 **75.0점**입니다.

각 `sample_id`는 내부에 포함된 detection 레코드 수와 관계없이 동일하게 **1개의 평가 케이스**로 계산됩니다.

In [ ]:
# 여기에 코드를 작성하세요.


## 결과 저장

채점을 위해, 위에서 구한 데이터를 현재 파일과 같은 디렉토리(.ipynb 파일이 있는 디렉토리)에 `submission.csv`이라는 이름으로 저장해 주세요.

In [ ]:
# csv 파일 저장 예시
df.to_csv('submission.csv', index=False)

## q1 풀이 코드와 결과

아래에는 실제 학습에 사용한 Python 소스, 선택된 모델/예측 파일, 검증 결과가 포함되어 있습니다. 기본 실행은 학습 과정에서 저장한 예측을 복원하고 제출 형식을 다시 검증합니다. RECOMPUTE=True로 바꾸면 저장 모델로 CPU 추론을 다시 실행합니다. 데이터 경로는 DATA_ROOT에서 조정하세요. 원본 문제 셀은 보존했습니다.

선택된 단계: `baseline`, 검증 지표: `accuracy`, 점수: `0.849`. 점수가 None이면 학습 완료 전의 기본값입니다.

이 경량본은 전체 풀이 소스와 실제 예측·검증 출력을 포함하지만 모델 파일은 포함하지 않습니다. 저장 예측 복원 모드이며, 학습이나 새 추론을 이 Notebook에서 했다고 표시하지 않습니다. 재학습/재추론에는 별도 모델과 의존 산출물이 필요할 수 있습니다.

### Python 소스: race90/models.py

```python
from pathlib import Path
import hashlib
import time
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, mean_squared_error, mean_absolute_error, log_loss, r2_score)
from .data import load_dataset
from .io import read_csv, make_submission

MINIMIZE = {"rmse", "mae", "log_loss"}


def split_indices(task, y, metadata, seed):
    split = task["split"]
    strategy = split["strategy"]
    fraction = float(split.get("fraction", 0.2))
    if not 0 < fraction < 1 or len(y) < 5:
        raise ValueError("Validation requires at least 5 records and 0 < fraction < 1")
    idx = np.arange(len(y))
    if strategy == "group":
        groups = metadata[split["group_column"]]
        if groups.nunique() < 2 or groups.isna().any():
            raise ValueError("Group split needs at least two complete groups")
        a, b = next(GroupShuffleSplit(n_splits=1, test_size=fraction, random_state=seed).split(idx, y, groups))
        if set(groups.iloc[a]) & set(groups.iloc[b]):
            raise AssertionError("Validation group leakage")
    elif strategy == "time":
        stamps = metadata[split["time_column"]]
        if stamps.isna().any():
            raise ValueError("Null validation timestamps")
        if not pd.api.types.is_numeric_dtype(stamps):
            stamps = pd.to_datetime(stamps, errors="raise", utc=True)
        order = np.argsort(stamps.to_numpy(), kind="stable")
        n_valid = max(1, int(np.ceil(len(idx) * fraction)))
        cutoff = stamps.iloc[order[-n_valid]]
        b = idx[(stamps >= cutoff).to_numpy()]
        a = idx[(stamps < cutoff).to_numpy()]
        gap = int(split.get("gap", 0))
        if gap < 0:
            raise ValueError("Validation gap must be non-negative")
        a = a[np.argsort(stamps.iloc[a].to_numpy(), kind="stable")]
        if gap:
            a = a[:-gap]
    elif strategy in {"random", "stratified"}:
        stratify = y if strategy == "stratified" and task["objective"] == "classification" else None
        a, b = train_test_split(idx, test_size=fraction, random_state=seed, stratify=stratify)
    else:
        raise ValueError(f"Unknown validation strategy: {strategy}")
    if len(a) < 2 or not len(b):
        raise ValueError("Empty/tiny training or validation partition")
    if task["objective"] == "classification" and set(y.iloc[b]) - set(y.iloc[a]):
        raise ValueError("Validation contains unseen classes; correct the split or collect more data")
    if task["objective"] == "classification" and y.iloc[a].nunique() < 2:
        raise ValueError("Classification training partition requires at least two classes")
    if task.get("metric") == "roc_auc" and y.iloc[b].nunique() != 2:
        raise ValueError("Binary ROC AUC validation requires both classes")
    fingerprint = hashlib.sha256(a.tobytes() + b.tobytes()).hexdigest()
    return a, b, fingerprint


def as_strings(x):
    frame = pd.DataFrame(x).astype(object)
    return frame.where(pd.notna(frame), "__missing__").astype(str).to_numpy()


def finite_values(x):
    frame = pd.DataFrame(x)
    return frame.replace([np.inf, -np.inf], np.nan)


def build_model(task, X, stage, threads, seed):
    classification = task["objective"] == "classification"
    if task["kind"] == "text":
        char = TfidfVectorizer(analyzer="char", ngram_range=(1, 5), dtype=np.float32,
                               max_features=task.get("max_char_features", 50000), sublinear_tf=True)
        features = char
        if stage == "upgrade":
            word = TfidfVectorizer(ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b", dtype=np.float32,
                                   max_features=30000, sublinear_tf=True)
            features = FeatureUnion([("char", char), ("word", word)])
        estimator = (LogisticRegression(C=task.get("C", 2.0), max_iter=250, random_state=seed)
                     if classification else Ridge(alpha=2.0, solver="lsqr"))
        return Pipeline([("features", features), ("model", estimator)])
    numeric = list(X.select_dtypes(include="number").columns)
    categorical = [c for c in X.columns if c not in numeric]
    histogram = stage == "upgrade" and task.get("cpu_estimator", "hist_gradient_boosting") == "hist_gradient_boosting"
    prep = ColumnTransformer([
        ("numeric", Pipeline([("finite", FunctionTransformer(finite_values)),
                               ("impute", SimpleImputer(strategy="median", keep_empty_features=True))]), numeric),
        ("category", Pipeline([("string", FunctionTransformer(as_strings)),
                                ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
                                 if histogram else OneHotEncoder(handle_unknown="ignore"))]), categorical)
    ], sparse_threshold=0.0 if histogram else 1.0)
    if histogram:
        # Avoid an internal random validation split for grouped/temporal tasks.
        kwargs = dict(max_iter=int(task.get("boosting_iterations", 160)), early_stopping=False,
                      learning_rate=float(task.get("learning_rate", .08)),
                      categorical_features=None, random_state=seed)
        estimator = HistGradientBoostingClassifier(**kwargs) if classification else HistGradientBoostingRegressor(**kwargs)
        return Pipeline([("preprocess", prep), ("model", estimator)])
    kwargs = dict(n_estimators=64 if stage == "baseline" else int(task.get("trees", 256)),
                  n_jobs=threads, random_state=seed, min_samples_leaf=1)
    estimator = ExtraTreesClassifier(class_weight="balanced", **kwargs) if classification else ExtraTreesRegressor(**kwargs)
    return Pipeline([("preprocess", prep), ("model", estimator)])


def predictions(model, X, task):
    if task.get("prediction_type", "label") != "probability":
        return model.predict(X)
    probs = model.predict_proba(X)
    classes = list(model.classes_)
    if len(task["prediction_columns"]) == 1:
        if "positive_class" not in task:
            raise ValueError("Set positive_class explicitly for a binary probability submission")
        return probs[:, classes.index(task["positive_class"])]
    order = task.get("class_order")
    if order is None or set(order) != set(classes):
        raise ValueError("class_order must explicitly map every class to probability columns")
    return probs[:, [classes.index(c) for c in order]]


def metric_score(model, X, y, task):
    metric = task["metric"]
    if metric == "roc_auc":
        if len(model.classes_) != 2 or task["positive_class"] not in model.classes_:
            raise ValueError("roc_auc requires two classes and a valid positive_class")
        return float(roc_auc_score((y == task["positive_class"]).astype(int),
                                  model.predict_proba(X)[:, list(model.classes_).index(task["positive_class"])]))
    if metric == "log_loss":
        return float(log_loss(y, model.predict_proba(X), labels=model.classes_))
    pred = model.predict(X)
    funcs = {"accuracy": accuracy_score, "balanced_accuracy": balanced_accuracy_score,
             "macro_f1": lambda a, b: f1_score(a, b, average="macro", zero_division=0),
             "weighted_f1": lambda a, b: f1_score(a, b, average="weighted", zero_division=0),
             "f1": lambda a, b: f1_score(a, b, pos_label=task["positive_class"], zero_division=0),
             "rmse": lambda a, b: np.sqrt(mean_squared_error(a, b)), "mae": mean_absolute_error, "r2": r2_score}
    if metric not in funcs:
        raise ValueError(f"Unsupported metric {metric}")
    return float(funcs[metric](y, pred))


def publish_fallback(ctx):
    task, cfg = ctx.task, ctx.cfg
    test = read_csv(cfg, task["test_csv"], task)
    ids = test[task["id_column"]].drop_duplicates().reset_index(drop=True)
    if task["kind"] in {"yolo", "custom"}:
        if task["kind"] == "custom":
            return  # No invented solution for an unknown task.
        if not task.get("allow_empty_predictions", False):
            raise ValueError("YOLO requires explicit allow_empty_predictions for no-detection rows")
        frame = make_submission(cfg, task, ids, np.array([""] * len(ids), dtype=object))
        ctx.publish(frame, None, {"algorithm": "empty_detections", "feature_mode": "empty"}, stage="fallback")
        return
    train = read_csv(cfg, task["train_csv"], task)
    y = train[task["target"]]
    if task["kind"] == "timeseries":
        y = train.groupby(task["id_column"], sort=False)[task["target"]].first()
    if y.isna().any():
        raise ValueError("Null targets")
    model = DummyClassifier(strategy="prior") if task["objective"] == "classification" else DummyRegressor(strategy="mean")
    model.fit(np.zeros((len(y), 1)), y)
    frame = make_submission(cfg, task, ids, predictions(model, np.zeros((len(ids), 1)), task))
    ctx.publish(frame, None, {"algorithm": type(model).__name__, "feature_mode": "constant"},
                model=model, stage="fallback")


def train_cpu(ctx):
    X, Xt, y, ids, metadata = load_dataset(ctx.cfg, ctx.task)
    fit_cpu_dataset(ctx, X, Xt, y, ids, metadata)


def fit_cpu_dataset(ctx, X, Xt, y, ids, metadata, model=None, details=None, artifacts=None):
    a, b, split_hash = split_indices(ctx.task, y, metadata, ctx.cfg["seed"])
    model = model if model is not None else build_model(ctx.task, X, ctx.stage, ctx.threads, ctx.cfg["seed"])
    started = time.monotonic()
    model.fit(X.iloc[a], y.iloc[a])
    fit_seconds = time.monotonic() - started
    score = metric_score(model, X.iloc[b], y.iloc[b], ctx.task)
    details = {"algorithm": type(model[-1]).__name__, "feature_mode": "dataset",
               "split_hash": split_hash, "train_rows": len(a), "valid_rows": len(b), **(details or {})}
    # Commit a real trained model before risking another full fit or a timeout.
    started = time.monotonic()
    frame = make_submission(ctx.cfg, ctx.task, ids, predictions(model, Xt, ctx.task))
    ctx.publish(frame, score, {**details, "refit_all_rows": False}, model=model, artifacts=artifacts)
    publish_seconds = time.monotonic() - started
    estimated_refit = fit_seconds * len(y) / len(a) * 1.5 + publish_seconds * 1.5 + 2
    if not ctx.task.get("refit", True) or ctx.seconds_left() < estimated_refit:
        return
    # The score remains the held-out score measured before refitting.
    model.fit(X, y)
    frame = make_submission(ctx.cfg, ctx.task, ids, predictions(model, Xt, ctx.task))
    ctx.publish(frame, score, {**details, "refit_all_rows": True}, model=model, artifacts=artifacts)


def predict_checkpoint(cfg, task, candidate, device="cpu"):
    import json
    candidate = Path(candidate)
    info = json.loads((candidate / "manifest.json").read_text())
    mode = info["details"]["feature_mode"]
    test = read_csv(cfg, task["test_csv"], task)
    ids = test[task["id_column"]].drop_duplicates().reset_index(drop=True)
    if mode == "empty":
        return make_submission(cfg, task, ids, np.array([""] * len(ids), dtype=object))
    if mode == "catboost":
        from .tabular_gpu import predict_tabular_checkpoint
        return predict_tabular_checkpoint(cfg, task, candidate)
    if mode in {"resnet", "resnet_features", "yolo"}:
        from .vision import predict_vision_checkpoint
        return predict_vision_checkpoint(cfg, task, candidate, device)
    if mode in {"sensor_cnn", "transformer"}:
        from .neural import predict_neural_checkpoint
        return predict_neural_checkpoint(cfg, task, candidate, device)
    model = joblib.load(candidate / "model.joblib")
    if mode == "constant":
        Xt = np.zeros((len(ids), 1))
    else:
        _, Xt, _, ids, _ = load_dataset(cfg, task)
    return make_submission(cfg, task, ids, predictions(model, Xt, task))

```

### Python 소스: race90/io.py

```python
from pathlib import Path
import hashlib
import json
import os
import uuid
import numpy as np
import pandas as pd
from .config import resolve_data


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)


def read_csv(cfg, path, task):
    kwargs = {"encoding": task.get("encoding", "utf-8"), **task.get("read_csv_kwargs", {})}
    return pd.read_csv(resolve_data(cfg, path), **kwargs)


def read_saved_submission(path, template, task, manifest=None):
    # Output CSVs are UTF-8 even when input data uses another encoding. Preserve
    # string labels such as "001" and all non-prediction columns on round trips.
    dtype = {c: template[c].dtype for c in template if c not in task["prediction_columns"]}
    for column, kind in (manifest or {}).get("submission_dtypes", {}).items():
        if column in task["prediction_columns"]:
            dtype[column] = kind
    return pd.read_csv(path, dtype=dtype, keep_default_na=False)


def validate_submission(frame, template, task):
    id_col = task["id_column"]
    targets = task["prediction_columns"]
    if list(frame.columns) != list(template.columns):
        raise ValueError("Submission column names/order differ from sample_submission")
    if len(frame) != len(template):
        raise ValueError("Submission row count differs from sample_submission")
    if id_col not in template or template[id_col].isna().any() or template[id_col].duplicated().any():
        raise ValueError("Sample submission IDs must be non-null and unique")
    if not frame[id_col].equals(template[id_col]):
        raise ValueError("Submission IDs/order differ from sample_submission")
    for col in targets:
        if col not in frame:
            raise ValueError(f"Missing prediction column {col}")
        if frame[col].isna().any():
            raise ValueError(f"Null predictions in {col}")
        if not task.get("allow_empty_predictions", False) and frame[col].astype(str).eq("").any():
            raise ValueError(f"Empty predictions in {col}")
        if pd.api.types.is_numeric_dtype(frame[col]) and not np.isfinite(frame[col]).all():
            raise ValueError(f"Non-finite predictions in {col}")
        if task.get("prediction_type") == "probability":
            p = pd.to_numeric(frame[col], errors="raise")
            if not p.between(0, 1).all():
                raise ValueError("Probabilities must lie in [0, 1]")
        if task.get("allowed_labels") is not None:
            if not frame[col].isin(task["allowed_labels"]).all():
                raise ValueError("Predictions contain labels outside allowed_labels")
    if task.get("prediction_type") == "probability" and len(targets) > 1:
        if not np.allclose(frame[targets].sum(axis=1), 1, atol=1e-5):
            raise ValueError("Multiclass probabilities must sum to one")
    untouched = [c for c in template.columns if c not in targets]
    if not frame[untouched].equals(template[untouched]):
        raise ValueError("Non-prediction columns changed")
    return frame


def make_submission(cfg, task, ids, predictions):
    template = read_csv(cfg, task["sample_submission"], task)
    id_col = task["id_column"]
    pred = np.asarray(predictions)
    if pred.ndim == 1:
        pred = pred[:, None]
    if pred.shape != (len(ids), len(task["prediction_columns"])):
        raise ValueError(f"Prediction shape {pred.shape} doesn't match configured output")
    series = pd.Series(ids).reset_index(drop=True)
    if series.isna().any() or series.duplicated().any():
        raise ValueError("Test IDs must be non-null and unique after feature extraction")
    if set(series.tolist()) != set(template[id_col].tolist()):
        raise ValueError("Test IDs and sample submission IDs differ")
    indexed = pd.DataFrame(pred, columns=task["prediction_columns"], index=series)
    ordered = indexed.loc[template[id_col]].reset_index(drop=True)
    result = template.copy()
    for c in task["prediction_columns"]:
        result[c] = ordered[c].to_numpy()
    return validate_submission(result, template, task)


def save_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()

```

### Python 소스: race90/vision.py

```python
from pathlib import Path
import time
import numpy as np
import pandas as pd
from .config import resolve_data
from .data import image_path
from .io import read_csv, make_submission
from .models import split_indices, metric_score


def normalized_yolo_yaml(cfg, task, output):
    """Make YAML/txt dataset paths portable between local files and the Pod."""
    import yaml
    source = resolve_data(cfg, task["data_yaml"])
    data = yaml.safe_load(source.read_text())
    root_spec = Path(data.get("path", "."))
    if root_spec.is_absolute():
        raise ValueError("YOLO YAML path must be relative to its YAML file for remote portability")
    root = (source.parent / root_spec).resolve()
    data_root = Path(cfg["data_root"]).resolve()
    if not root.is_relative_to(data_root):
        raise ValueError("YOLO dataset escapes data_root")
    split_images = {}
    for split in ("train", "val"):
        entries = data.get(split)
        if not entries:
            raise ValueError(f"YOLO dataset requires an explicit {split} split")
        entries = entries if isinstance(entries, list) else [entries]
        resolved, images = [], []
        for entry in entries:
            relative = Path(entry)
            if relative.is_absolute():
                raise ValueError("YOLO split paths must be relative to the dataset path")
            path = (root / relative).resolve()
            if not path.is_relative_to(data_root) or not path.exists():
                raise ValueError(f"Invalid YOLO {split} path: {entry}")
            if path.is_dir():
                images.extend(str(p.resolve()) for p in path.rglob("*")
                              if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"})
                resolved.append(str(path))
            elif path.suffix == ".txt":
                listed = []
                for line in path.read_text().splitlines():
                    if not line.strip():
                        continue
                    item = Path(line.strip())
                    if item.is_absolute():
                        raise ValueError("YOLO image lists must use paths relative to the txt file")
                    item = (path.parent / item).resolve()
                    if not item.is_relative_to(data_root) or not item.is_file():
                        raise ValueError(f"Invalid image list entry: {line}")
                    listed.append(str(item))
                listing = Path(output) / f"{split}-{len(resolved)}.runtime.txt"
                listing.parent.mkdir(parents=True, exist_ok=True)
                listing.write_text("\n".join(listed) + "\n")
                resolved.append(str(listing))
                images.extend(listed)
            else:
                raise ValueError("YOLO train/val must be an image directory or txt image list")
        if not images or len(set(images)) != len(images):
            raise ValueError(f"YOLO {split} split is empty or contains duplicate images")
        split_images[split] = set(images)
        data[split] = resolved if len(resolved) > 1 else resolved[0]
    if split_images["train"] & split_images["val"]:
        raise ValueError("YOLO training and validation images overlap")
    data["path"] = str(root)
    runtime = Path(output) / "dataset.runtime.yaml"
    runtime.parent.mkdir(parents=True, exist_ok=True)
    runtime.write_text(yaml.safe_dump(data, sort_keys=False))
    return runtime, {split: len(images) for split, images in split_images.items()}


def torch_components():
    import torch
    from torchvision import models, transforms
    return torch, models, transforms


def image_loader(cfg, task, frame, labels=None, train=False, shuffle=False):
    torch, _, T = torch_components()
    from PIL import Image
    size = int(task.get("image_size", 224))
    ops = ([T.RandomResizedCrop(size, scale=(0.7, 1.0)), T.RandomHorizontalFlip()]
           if train else [T.Resize(size + 32), T.CenterCrop(size)])
    transform = T.Compose(ops + [T.ToTensor(), T.Normalize([.485, .456, .406], [.229, .224, .225])])

    class Images(torch.utils.data.Dataset):
        def __len__(self):
            return len(frame)

        def __getitem__(self, idx):
            with Image.open(image_path(cfg, task, frame.iloc[idx][task["image_column"]])) as im:
                x = transform(im.convert("RGB"))
            return x, (int(labels[idx]) if labels is not None else 0)

    # Dataset is local to this function; workers=0 also avoids extra CPU contention.
    return torch.utils.data.DataLoader(Images(), batch_size=int(task.get("batch_size", 32)),
                                      shuffle=shuffle, num_workers=0, pin_memory=torch.cuda.is_available())


def image_probs(net, loader, device):
    torch, _, _ = torch_components()
    net.eval()
    output = []
    with torch.inference_mode():
        for images, _ in loader:
            output.append(net(images.to(device)).softmax(1).cpu().numpy())
    return np.concatenate(output)


class Probabilities:
    def __init__(self, probs, classes):
        self.probs, self.classes_ = probs, np.asarray(classes)

    def predict(self, _):
        return self.classes_[self.probs.argmax(axis=1)]

    def predict_proba(self, _):
        return self.probs


def output_probs(cfg, task, ids, probs, classes):
    from .models import predictions
    return make_submission(cfg, task, ids, predictions(Probabilities(probs, classes), None, task))


def train_resnet(ctx):
    torch, models, _ = torch_components()
    torch.set_num_threads(ctx.threads)
    torch.manual_seed(ctx.cfg["seed"])
    np.random.seed(ctx.cfg["seed"])
    if ctx.device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable; baseline is retained")
    task = dict(ctx.task)
    train = read_csv(ctx.cfg, task["train_csv"], task)
    test = read_csv(ctx.cfg, task["test_csv"], task)
    y = train[task["target"]]
    classes = sorted(y.unique().tolist())
    if task["objective"] != "classification":
        raise ValueError("Built-in image adapter supports classification; use custom for regression")
    a, b, split_hash = split_indices(task, y, train, ctx.cfg["seed"])
    labels = np.array([classes.index(v) for v in y])
    weights = models.ResNet18_Weights.DEFAULT if task.get("pretrained", True) else None
    net = models.resnet18(weights=weights)
    frozen = task.get("freeze_backbone", True)
    if frozen:
        for p in net.parameters():
            p.requires_grad = False
    net.fc = torch.nn.Linear(net.fc.in_features, len(classes))
    net.to(ctx.device)
    optimizer = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad],
                                 lr=float(task.get("learning_rate", 1e-3)))
    use_amp = ctx.device.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best = None
    checkpoint = ctx.output / "resnet-current.pt"
    for epoch in range(int(task.get("epochs", 8))):
        if ctx.seconds_left() < task.get("inference_reserve_seconds", 90):
            break
        loader = image_loader(ctx.cfg, task, train.iloc[a].reset_index(drop=True), labels[a], train=True, shuffle=True)
        net.train()
        if frozen:
            net.eval()  # Keep frozen BatchNorm statistics unchanged.
            net.fc.train()
        try:
            for images, targets in loader:
                if ctx.seconds_left() < task.get("inference_reserve_seconds", 90):
                    break
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda" if use_amp else "cpu", enabled=use_amp):
                    loss = torch.nn.functional.cross_entropy(net(images.to(ctx.device)), targets.to(ctx.device))
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        except torch.cuda.OutOfMemoryError:
            if task.get("batch_size", 32) <= 1:
                raise
            task["batch_size"] = max(1, task.get("batch_size", 32) // 2)
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            continue
        val_loader = image_loader(ctx.cfg, task, train.iloc[b].reset_index(drop=True))
        probs = image_probs(net, val_loader, ctx.device)
        score = metric_score(Probabilities(probs, classes), None, y.iloc[b], task)
        from .models import MINIMIZE
        if best is not None and (score >= best if task["metric"] in MINIMIZE else score <= best):
            continue
        test_probs = image_probs(net, image_loader(ctx.cfg, task, test), ctx.device)
        frame = output_probs(ctx.cfg, task, test[task["id_column"]], test_probs, classes)
        torch.save({"state_dict": net.state_dict(), "classes": classes}, checkpoint)
        ctx.publish(frame, score, {"algorithm": "resnet18", "feature_mode": "resnet", "epoch": epoch + 1,
                                  "split_hash": split_hash, "train_rows": len(a), "valid_rows": len(b),
                                  "pretrained": bool(weights), "refit_all_rows": False},
                    artifacts={"model.pt": checkpoint})
        best = score
    if best is None:
        raise RuntimeError("No vision epoch completed within the deadline")


def yolo_predictions(cfg, task, weights, device):
    from ultralytics import YOLO
    from .worker import callable_from
    test = read_csv(cfg, task["test_csv"], task)
    paths = [str(image_path(cfg, task, v)) for v in test[task["image_column"]]]
    model = YOLO(str(weights))
    results = model.predict(paths, device=device, imgsz=int(task.get("image_size", 640)),
                            batch=int(task.get("batch_size", 8)), stream=True,
                            conf=float(task.get("confidence", 0.25)), verbose=False)
    if task.get("exporter"):
        return callable_from(task["exporter"])(cfg, task, test, results)
    # Explicit built-in format: one row per image, space-separated
    # class confidence xmin ymin xmax ymax blocks, absolute pixel coordinates.
    if task.get("detection_format") != "class_conf_xyxy":
        raise ValueError("Set detection_format=class_conf_xyxy or provide exporter=module:function")
    strings = []
    for result in results:
        boxes = result.boxes
        tokens = []
        if boxes is not None:
            for cls, conf, xyxy in zip(boxes.cls.cpu().tolist(), boxes.conf.cpu().tolist(), boxes.xyxy.cpu().tolist()):
                tokens.extend([str(int(cls)), f"{conf:.6f}", *(f"{v:.2f}" for v in xyxy)])
        strings.append(" ".join(tokens))
    return make_submission(cfg, task, test[task["id_column"]], np.array(strings, dtype=object))


def train_yolo(ctx):
    from ultralytics import YOLO
    task = ctx.task
    model = YOLO(task.get("weights", "yolo11n.pt"))
    best = [None]
    def publish_epoch(trainer):
        # on_model_save runs after checkpoint creation and metric calculation.
        if ctx.seconds_left() < task.get("inference_reserve_seconds", 90):
            trainer.stop = True
            return
        metric_key = "metrics/mAP50(B)" if task["metric"] == "map50" else "metrics/mAP50-95(B)"
        score = trainer.metrics.get(metric_key)
        if score is None or (best[0] is not None and score <= best[0]) or not Path(trainer.last).exists():
            return
        # Current epoch metrics belong to last.pt; best.pt can be from a different
        # epoch when the competition metric differs from YOLO's own fitness.
        frame = yolo_predictions(ctx.cfg, task, trainer.last, ctx.device)
        ctx.publish(frame, float(score), {"algorithm": "YOLO", "feature_mode": "yolo", "epoch": trainer.epoch + 1,
                                          "validation": task["data_yaml"], "refit_all_rows": False},
                    artifacts={"model.pt": trainer.last})
        best[0] = float(score)
    model.add_callback("on_model_save", publish_epoch)
    train_seconds = max(1, ctx.seconds_left() - task.get("inference_reserve_seconds", 90))
    dataset_yaml, _ = normalized_yolo_yaml(ctx.cfg, task, ctx.output)
    model.train(data=str(dataset_yaml), device=ctx.device,
                epochs=int(task.get("epochs", 30)), time=train_seconds / 3600,
                imgsz=int(task.get("image_size", 640)), batch=int(task.get("batch_size", 8)),
                workers=0, seed=ctx.cfg["seed"], patience=5, save=True,
                project=str(ctx.output / "yolo_training"), name="train", exist_ok=True, verbose=False)
    if best[0] is None:
        raise RuntimeError("YOLO produced no validated checkpoint; fallback retained")


def train_vision(ctx):
    if ctx.task["kind"] == "yolo":
        train_yolo(ctx)
    else:
        train_resnet(ctx)


def resnet_embeddings(cfg, task, frame, net, device="cpu"):
    """One forward pass per image; the backbone is never trained on exam data."""
    torch, _, _ = torch_components()
    net.eval()
    rows = []
    with torch.inference_mode():
        for images, _ in image_loader(cfg, task, frame):
            rows.append(net(images.to(device)).cpu().numpy())
    return pd.DataFrame(np.concatenate(rows))


def train_image_embeddings(ctx):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from .models import fit_cpu_dataset
    torch, models, _ = torch_components()
    torch.set_num_threads(ctx.threads)
    torch.manual_seed(ctx.cfg["seed"])
    task = ctx.task
    if task["objective"] != "classification":
        raise ValueError("Image embeddings currently support classification")
    weights = models.ResNet18_Weights.DEFAULT if task.get("pretrained", True) else None
    net = models.resnet18(weights=weights)
    net.fc = torch.nn.Identity()
    train = read_csv(ctx.cfg, task["train_csv"], task)
    test = read_csv(ctx.cfg, task["test_csv"], task)
    X = resnet_embeddings(ctx.cfg, task, train, net)
    Xt = resnet_embeddings(ctx.cfg, task, test, net)
    checkpoint = ctx.output / "backbone-current.pt"
    torch.save(net.state_dict(), checkpoint)
    classifier = make_pipeline(StandardScaler(), LogisticRegression(C=task.get("C", 1.0), max_iter=300,
                                                                    random_state=ctx.cfg["seed"]))
    fit_cpu_dataset(ctx, X, Xt, train[task["target"]], test[task["id_column"]], train,
                    model=classifier, details={"feature_mode": "resnet_features", "pretrained": bool(weights)},
                    artifacts={"backbone.pt": checkpoint})


def predict_vision_checkpoint(cfg, task, candidate, device):
    if task["kind"] == "yolo":
        return yolo_predictions(cfg, task, candidate / "model.pt", device)
    torch, models, _ = torch_components()
    if (candidate / "backbone.pt").exists():
        import joblib
        from .models import predictions
        net = models.resnet18(weights=None)
        net.fc = torch.nn.Identity()
        net.load_state_dict(torch.load(candidate / "backbone.pt", map_location="cpu", weights_only=True))
        net.to(device)
        test = read_csv(cfg, task["test_csv"], task)
        X = resnet_embeddings(cfg, task, test, net, device)
        classifier = joblib.load(candidate / "model.joblib")
        return make_submission(cfg, task, test[task["id_column"]], predictions(classifier, X, task))
    checkpoint = torch.load(candidate / "model.pt", map_location="cpu", weights_only=True)
    net = models.resnet18(weights=None)
    net.fc = torch.nn.Linear(net.fc.in_features, len(checkpoint["classes"]))
    net.load_state_dict(checkpoint["state_dict"])
    net.to(device)
    test = read_csv(cfg, task["test_csv"], task)
    probs = image_probs(net, image_loader(cfg, task, test), device)
    return output_probs(cfg, task, test[task["id_column"]], probs, checkpoint["classes"])

```

### Python 소스: race90/config.py

```python
from pathlib import Path
import math
import re
import yaml

KINDS = {"tabular", "text", "timeseries", "image", "yolo", "custom"}
CLASSIFICATION_METRICS = {"accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "f1", "roc_auc", "log_loss"}
REGRESSION_METRICS = {"rmse", "mae", "r2"}


def load_config(path):
    path = Path(path).resolve()
    cfg = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(cfg, dict):
        raise ValueError("Configuration must be a mapping")
    cfg.setdefault("total_minutes", 90)
    cfg.setdefault("finalize_minutes", 20)
    cfg.setdefault("cpu_workers", 3)
    cfg.setdefault("seed", 42)
    cfg.setdefault("data_root", "data")
    cfg.setdefault("output_root", "runs")
    cfg.setdefault("runpod", {"enabled": False})
    if any(k in cfg["runpod"] for k in ("api_key", "apiKey", "RUNPOD_API_KEY")):
        raise ValueError("Keep API keys in RUNPOD_API_KEY/environment, not the configuration file")
    cfg.setdefault("notebooks", {"execute": True, "embed_models": True})
    for key in ("data_root", "output_root"):
        p = Path(cfg[key]).expanduser()
        cfg[key] = str((path.parent / p).resolve() if not p.is_absolute() else p.resolve())
    cfg["project_root"] = str(path.parent)
    if not all(math.isfinite(float(cfg[k])) for k in ("total_minutes", "finalize_minutes")) or not 0 < cfg["finalize_minutes"] < cfg["total_minutes"]:
        raise ValueError("0 < finalize_minutes < total_minutes required")
    from .timing import timeline
    plan = timeline(cfg, 0)
    if any(not math.isfinite(plan[k]) or plan[k] < 0 for k in ("upload_seconds", "retrieval_seconds", "pack_seconds")):
        raise ValueError("Time management reserves must be finite and non-negative")
    if plan["retrieval_deadline"] >= plan["notebook_deadline"]:
        raise ValueError("Finalization reserve must leave time for Notebook execution after retrieval, packing and upload")
    minimum = float(cfg.get("time_management", {}).get("minimum_upgrade_seconds", 0))
    if not math.isfinite(minimum) or minimum < 0:
        raise ValueError("minimum_upgrade_seconds must be finite and non-negative")
    if int(cfg["cpu_workers"]) < 1:
        raise ValueError("cpu_workers must be positive")
    if cfg.get("cpu_threads") is not None and int(cfg["cpu_threads"]) < 1:
        raise ValueError("cpu_threads must be positive")
    if int(cfg.get("notebook_workers", 1)) < 1:
        raise ValueError("notebook_workers must be positive")
    if cfg["runpod"].get("enabled") and int(cfg["runpod"].get("gpu_count", 1)) < 1:
        raise ValueError("runpod.gpu_count must be positive")
    if cfg["notebooks"].get("mode", "restore") not in {"restore", "infer"}:
        raise ValueError("notebooks.mode must be restore or infer")
    if cfg["notebooks"].get("mode", "restore") == "infer" and not cfg["notebooks"].get("embed_models", True):
        raise ValueError("notebooks.mode=infer requires embed_models=true")
    tasks = cfg.get("tasks", [])
    if not tasks:
        raise ValueError("At least one task is required")
    seen = set()
    for t in tasks:
        name = t.get("name", "")
        if not re.fullmatch(r"[A-Za-z0-9_-]+", name) or name in seen:
            raise ValueError(f"Unique, simple task name required: {name!r}")
        seen.add(name)
        if t.get("kind") not in KINDS:
            raise ValueError(f"Unsupported task kind: {t.get('kind')}")
        for key in ("test_csv", "sample_submission", "id_column", "prediction_columns"):
            if not t.get(key):
                raise ValueError(f"{name}: {key} is required")
        if not isinstance(t["prediction_columns"], list) or t["id_column"] in t["prediction_columns"]:
            raise ValueError(f"{name}: prediction_columns must be a list excluding the ID column")
        if len(set(t["prediction_columns"])) != len(t["prediction_columns"]):
            raise ValueError(f"{name}: duplicate prediction_columns")
        if t["kind"] not in {"yolo", "custom"}:
            for key in ("train_csv", "target"):
                if not t.get(key):
                    raise ValueError(f"{name}: {key} is required")
        if t["kind"] == "custom" and not t.get("adapter"):
            raise ValueError(f"{name}: adapter module:function required")
        if t["kind"] == "text" and not t.get("text_columns"):
            raise ValueError(f"{name}: text_columns required")
        if t["kind"] == "timeseries" and not t.get("sensor_columns"):
            raise ValueError(f"{name}: sensor_columns required")
        if t["kind"] in {"image", "yolo"} and not t.get("image_column"):
            raise ValueError(f"{name}: image_column required")
        t.setdefault("objective", "classification")
        t.setdefault("metric", "map50" if t["kind"] == "yolo" else
                     ("accuracy" if t["objective"] == "classification" else "rmse"))
        if t["objective"] not in {"classification", "regression"}:
            raise ValueError(f"{name}: objective must be classification or regression")
        t.setdefault("prediction_type", "label")
        t.setdefault("resource", "gpu" if t["kind"] in {"image", "yolo"} else "cpu")
        if t["resource"] not in {"cpu", "gpu"}:
            raise ValueError(f"{name}: resource must be cpu or gpu")
        if t.get("cpu_estimator", "hist_gradient_boosting") not in {"hist_gradient_boosting", "extra_trees"}:
            raise ValueError(f"{name}: unknown cpu_estimator")
        if t.get("image_upgrade", "trees") not in {"trees", "embeddings"}:
            raise ValueError(f"{name}: image_upgrade must be trees or embeddings")
        t.setdefault("depends_on", [])
        if not isinstance(t["depends_on"], list):
            raise ValueError(f"{name}: depends_on must be a list")
        t.setdefault("split", {"strategy": "stratified", "fraction": 0.2})
        split = t["split"]
        if split.get("strategy") not in {"stratified", "random", "group", "time"}:
            raise ValueError(f"{name}: unknown split strategy")
        if not 0 < float(split.get("fraction", .2)) < 1:
            raise ValueError(f"{name}: split fraction must be in (0, 1)")
        for strategy, column in (("group", "group_column"), ("time", "time_column")):
            if split["strategy"] == strategy and not split.get(column):
                raise ValueError(f"{name}: {column} is required for {strategy} split")
        if int(split.get("gap", 0)) != float(split.get("gap", 0)) or int(split.get("gap", 0)) < 0:
            raise ValueError(f"{name}: split gap must be a non-negative integer")
        for key in ("baseline_seconds", "upgrade_seconds"):
            if key in t and (not math.isfinite(float(t[key])) or float(t[key]) <= 0):
                raise ValueError(f"{name}: {key} must be positive and finite")
        if t["kind"] not in {"yolo", "custom"}:
            metrics = CLASSIFICATION_METRICS if t["objective"] == "classification" else REGRESSION_METRICS
            if t["metric"] not in metrics:
                raise ValueError(f"{name}: unsupported metric for {t['objective']}: {t['metric']}")
        if t["prediction_type"] not in {"label", "probability"}:
            raise ValueError(f"{name}: prediction_type must be label or probability")
        if t["metric"] in {"roc_auc", "f1"} and "positive_class" not in t:
            raise ValueError(f"{name}: {t['metric']} requires positive_class (binary metric)")
        if t["prediction_type"] == "probability":
            if t["objective"] != "classification":
                raise ValueError(f"{name}: probability requires classification")
            columns = t["prediction_columns"]
            if len(columns) == 1 and "positive_class" not in t:
                raise ValueError(f"{name}: binary probability requires positive_class")
            if len(columns) > 1 and (len(t.get("class_order", [])) != len(columns) or
                                    len(set(t.get("class_order", []))) != len(columns)):
                raise ValueError(f"{name}: class_order must map each probability column exactly once")
        if t["kind"] not in {"custom", "yolo"} and t["prediction_type"] == "label" and len(t["prediction_columns"]) != 1:
            raise ValueError(f"{name}: built-in label/regression models require one prediction column")
        if t["kind"] == "yolo" and t["metric"] not in {"map50", "map50_95"}:
            raise ValueError(f"{name}: YOLO metric must be map50 or map50_95")
        if t["kind"] == "timeseries" and t["split"]["strategy"] not in {"group", "time"}:
            raise ValueError(f"{name}: choose group or time split explicitly for sensor data")
        if t["kind"] == "yolo" and not t.get("data_yaml"):
            raise ValueError(f"{name}: data_yaml with explicit train/val split required")
        # All data stays under one root, including remote transfer and Notebook replay.
        for key in ("train_csv", "test_csv", "sample_submission", "notebook", "image_root", "data_yaml"):
            if t.get(key):
                resolve_data(cfg, t[key])
    names = {t["name"] for t in tasks}
    for t in tasks:
        if set(t["depends_on"]) - names or t["name"] in t["depends_on"]:
            raise ValueError(f"{t['name']}: invalid dependency")
        if t.get("dependency_features") and t["kind"] != "tabular":
            raise ValueError("dependency_features is for tabular tasks; use ctx.dependencies in custom adapters for other types")
        for feature in t.get("dependency_features", []):
            if feature["task"] not in t["depends_on"]:
                raise ValueError("dependency_features must name a declared depends_on task")
            for key in ("train_file", "test_file"):
                if Path(feature[key]).name != feature[key]:
                    raise ValueError("Dependency feature files must have simple filenames")
    # Stable topological order ensures CPU queues never put children before parents.
    ordered, pending = [], list(tasks)
    while pending:
        available = next((t for t in pending if set(t["depends_on"]) <= {x["name"] for x in ordered}), None)
        if available is None:
            raise ValueError("Task dependency cycle detected")
        ordered.append(available)
        pending.remove(available)
    cfg["tasks"] = ordered
    cfg["dependency_outputs"] = {}
    return cfg


def resolve_data(cfg, value):
    root = Path(cfg["data_root"]).resolve()
    value = Path(value)
    if value.is_absolute():
        raise ValueError(f"Data paths must be relative to data_root: {value}")
    result = (root / value).resolve()
    if not result.is_relative_to(root):
        raise ValueError(f"Data path escapes data_root: {value}")
    return result

```

### Python 소스: race90/worker.py

```python
from dataclasses import dataclass
from pathlib import Path
import importlib
import json
import shutil
import time
import traceback
import uuid
import joblib
import numpy as np
from .io import atomic_json, save_csv, validate_submission, read_csv, sha256


@dataclass
class Context:
    cfg: dict
    task: dict
    output: Path
    stage: str
    deadline: float
    threads: int = 1
    device: str = "cpu"

    def seconds_left(self):
        return max(0.0, self.deadline - time.time())

    @property
    def dependencies(self):
        """Immutable selected candidate directories of declared parent tasks."""
        return {name: Path(self.cfg["dependency_outputs"][name]) for name in self.task.get("depends_on", [])}

    def publish(self, frame, score, details, model=None, artifacts=None, stage=None):
        template = read_csv(self.cfg, self.task["sample_submission"], self.task)
        validate_submission(frame, template, self.task)
        if score is not None and not np.isfinite(score):
            raise ValueError("Non-finite validation score")
        name = f"{stage or self.stage}-{uuid.uuid4().hex[:10]}"
        candidate = self.output / "candidates" / name
        candidate.mkdir(parents=True)
        if model is not None:
            joblib.dump(model, candidate / "model.joblib", compress=3)
        for filename, src in (artifacts or {}).items():
            if Path(filename).name != filename:
                raise ValueError("Artifact filenames must be simple")
            shutil.copy2(src, candidate / filename)
        save_csv(candidate / "submission.csv", frame)
        manifest = {"task": self.task["name"], "stage": stage or self.stage, "score": score,
                    "candidate_id": name,
                    "dependencies": {parent: json.loads((directory / "manifest.json").read_text()).get("candidate_id", directory.name)
                                     for parent, directory in self.dependencies.items()},
                    "metric": self.task["metric"], "details": details, "created_at": time.time(),
                    "sha256": sha256(candidate / "submission.csv"), "rows": len(frame),
                    "submission_dtypes": {column: str(dtype) for column, dtype in frame.dtypes.items()},
                    "artifacts": [p.name for p in candidate.iterdir() if p.name != "submission.csv"]}
        # Manifest is the commit marker: interrupted candidates cannot be selected.
        atomic_json(candidate / "manifest.json", manifest)
        print(json.dumps({"event": "candidate", "task": self.task["name"], "stage": manifest["stage"],
                          "score": score, "candidate": name}), flush=True)
        return candidate


def callable_from(spec):
    module, name = spec.split(":", 1)
    return getattr(importlib.import_module(module), name)


def run_worker(payload):
    ctx = Context(payload["cfg"], payload["task"], Path(payload["output"]), payload["stage"],
                  payload["deadline"], payload.get("threads", 1), payload.get("device", "cpu"))
    ctx.output.mkdir(parents=True, exist_ok=True)
    try:
        if ctx.task["kind"] == "custom":
            callable_from(ctx.task["adapter"])(ctx)
        else:
            from .models import publish_fallback, train_cpu
            from .data import audit
            if ctx.stage == "baseline":
                publish_fallback(ctx)
                audit(ctx.cfg, ctx.task, ctx.output)
            else:
                # GPU stages can now start before the local audit finishes.
                # Check table/split structure on the worker as well.
                audit(ctx.cfg, ctx.task, ctx.output, verify_images=False)
            if ctx.task["kind"] == "yolo" and ctx.stage == "baseline":
                pass
            elif ctx.task["kind"] == "tabular" and ctx.stage == "upgrade" and ctx.device != "cpu":
                from .tabular_gpu import train_tabular
                train_tabular(ctx)
            elif ctx.task["kind"] in {"image", "yolo"} and ctx.stage == "upgrade":
                if ctx.task["kind"] == "image" and ctx.device == "cpu":
                    if ctx.task.get("image_upgrade", "trees") == "embeddings":
                        from .vision import train_image_embeddings
                        train_image_embeddings(ctx)
                    else:
                        train_cpu(ctx)
                else:
                    from .vision import train_vision
                    train_vision(ctx)
            elif ctx.stage == "upgrade" and ctx.device != "cpu" and ctx.task["kind"] in {"text", "timeseries"}:
                from .neural import train_text, train_sensor
                (train_text if ctx.task["kind"] == "text" else train_sensor)(ctx)
            else:
                train_cpu(ctx)
        atomic_json(ctx.output / f"{ctx.stage}-status.json", {"status": "done", "time": time.time()})
        return 0
    except Exception as e:
        atomic_json(ctx.output / f"{ctx.stage}-status.json", {"status": "failed", "error": str(e), "time": time.time()})
        traceback.print_exc()
        return 1

```

### Python 소스: race90/neural.py

```python
"""GPU sensor CNN and text transfer learning with CPU-compatible checkpoints."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
from .data import load_dataset, recording_times
from .io import read_csv, atomic_json
from .models import split_indices, metric_score, MINIMIZE
from .vision import Probabilities, output_probs


def sequence_tensor(frame, task):
    sensors, ident = task["sensor_columns"], task["id_column"]
    if task.get("time_column"):
        frame = frame.sort_values([ident, task["time_column"]], kind="stable")
    length = int(task.get("sequence_length", 256))
    ids, sequences, metadata = [], [], []
    for key, group in frame.groupby(ident, sort=False):
        values = group[sensors].to_numpy(dtype=np.float32)
        channels = []
        for col in values.T:
            valid = np.isfinite(col)
            if not valid.any():
                channels.append(np.zeros(length, dtype=np.float32))
            else:
                channels.append(np.interp(np.linspace(0, max(0, len(col) - 1), length),
                                          np.arange(len(col))[valid], col[valid]).astype(np.float32))
        sequences.append(channels)
        ids.append(key)
        metadata.append(group.iloc[0])
    return np.array(sequences, dtype=np.float32), pd.Series(ids), pd.DataFrame(metadata).reset_index(drop=True)


def sensor_net(channels, classes):
    import torch
    return torch.nn.Sequential(torch.nn.Conv1d(channels, 64, 5, padding=2), torch.nn.ReLU(),
                               torch.nn.Conv1d(64, 128, 5, padding=2), torch.nn.ReLU(),
                               torch.nn.AdaptiveAvgPool1d(1), torch.nn.Flatten(),
                               torch.nn.Linear(128, classes))


def tensor_probs(model, tensor, device, batch_size):
    import torch
    model.eval()
    output = []
    with torch.inference_mode():
        for start in range(0, len(tensor), batch_size):
            output.append(model(torch.as_tensor(tensor[start:start + batch_size], device=device)).softmax(1).cpu().numpy())
    return np.concatenate(output)


def train_sensor(ctx):
    import torch
    torch.set_num_threads(ctx.threads)
    torch.manual_seed(ctx.cfg["seed"])
    task = ctx.task
    if task["objective"] != "classification":
        raise ValueError("Built-in sensor CNN supports classification; use CPU or custom for regression")
    train = read_csv(ctx.cfg, task["train_csv"], task)
    X, _, metadata = sequence_tensor(train, task)
    metadata = recording_times(train, metadata, task)
    Xt, ids, _ = sequence_tensor(read_csv(ctx.cfg, task["test_csv"], task), task)
    y = metadata[task["target"]]
    classes = sorted(y.unique().tolist())
    labels = np.array([classes.index(v) for v in y])
    a, b, split_hash = split_indices(task, y, metadata, ctx.cfg["seed"])
    mean, std = X[a].mean(axis=(0, 2), keepdims=True), X[a].std(axis=(0, 2), keepdims=True).clip(1e-6)
    X, Xt = (X - mean) / std, (Xt - mean) / std
    model = sensor_net(len(task["sensor_columns"]), len(classes)).to(ctx.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(task.get("learning_rate", 1e-3)))
    batch = int(task.get("batch_size", 128))
    rng = np.random.default_rng(ctx.cfg["seed"])
    best, stale = None, 0
    checkpoint = ctx.output / "sensor-current.pt"
    for epoch in range(int(task.get("epochs", 30))):
        if ctx.seconds_left() < task.get("inference_reserve_seconds", 60):
            break
        order = rng.permutation(a)
        model.train()
        for start in range(0, len(order), batch):
            if ctx.seconds_left() < task.get("inference_reserve_seconds", 60):
                break
            idx = order[start:start + batch]
            optimizer.zero_grad(set_to_none=True)
            loss = torch.nn.functional.cross_entropy(model(torch.as_tensor(X[idx], device=ctx.device)),
                                                      torch.as_tensor(labels[idx], device=ctx.device))
            loss.backward()
            optimizer.step()
        probs = tensor_probs(model, X[b], ctx.device, batch)
        score = metric_score(Probabilities(probs, classes), None, y.iloc[b], task)
        improved = best is None or (score < best if task["metric"] in MINIMIZE else score > best)
        if not improved:
            stale += 1
            if stale >= int(task.get("patience", 5)):
                break
            continue
        stale = 0
        frame = output_probs(ctx.cfg, task, ids, tensor_probs(model, Xt, ctx.device, batch), classes)
        torch.save({"state_dict": model.state_dict(), "classes": classes,
                    "mean": torch.tensor(mean), "std": torch.tensor(std)}, checkpoint)
        ctx.publish(frame, score, {"algorithm": "sensor_cnn", "feature_mode": "sensor_cnn", "epoch": epoch + 1,
                                  "split_hash": split_hash, "refit_all_rows": False}, artifacts={"model.pt": checkpoint})
        best = score
    if best is None:
        raise RuntimeError("No sensor CNN checkpoint completed within budget")


def text_batches(tokenizer, texts, indices, device, task):
    batch = int(task.get("batch_size", 32))
    for start in range(0, len(indices), batch):
        idx = indices[start:start + batch]
        encoded = tokenizer(texts.iloc[idx].tolist(), padding=True, truncation=True,
                            max_length=int(task.get("max_length", 128)), return_tensors="pt")
        yield idx, {k: v.to(device) for k, v in encoded.items()}


def text_probs(model, tokenizer, texts, device, task):
    import torch
    model.eval()
    out = []
    with torch.inference_mode():
        for _, batch in text_batches(tokenizer, texts, np.arange(len(texts)), device, task):
            out.append(model(**batch).logits.softmax(1).cpu().numpy())
    return np.concatenate(out)


def train_text(ctx):
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    torch.set_num_threads(ctx.threads)
    torch.manual_seed(ctx.cfg["seed"])
    task = ctx.task
    if task["objective"] != "classification":
        raise ValueError("Built-in transformer supports classification; use CPU or custom for regression")
    X, Xt, y, ids, metadata = load_dataset(ctx.cfg, task)
    a, b, split_hash = split_indices(task, y, metadata, ctx.cfg["seed"])
    classes = sorted(y.unique().tolist())
    labels = np.array([classes.index(v) for v in y])
    model_name = task.get("model_name", "klue/roberta-small")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(classes),
                                                               ignore_mismatched_sizes=True).to(ctx.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(task.get("learning_rate", 2e-5)))
    amp = ctx.device.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    rng = np.random.default_rng(ctx.cfg["seed"])
    best = None
    current = ctx.output / "transformer-current"
    current.mkdir(exist_ok=True)
    for epoch in range(int(task.get("epochs", 3))):
        if ctx.seconds_left() < task.get("inference_reserve_seconds", 120):
            break
        model.train()
        for idx, encoded in text_batches(tokenizer, X, rng.permutation(a), ctx.device, task):
            if ctx.seconds_left() < task.get("inference_reserve_seconds", 120):
                break
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda" if amp else "cpu", enabled=amp):
                loss = model(**encoded, labels=torch.as_tensor(labels[idx], device=ctx.device)).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        score = metric_score(Probabilities(text_probs(model, tokenizer, X.iloc[b].reset_index(drop=True), ctx.device, task), classes),
                             None, y.iloc[b], task)
        if best is not None and (score >= best if task["metric"] in MINIMIZE else score <= best):
            continue
        frame = output_probs(ctx.cfg, task, ids, text_probs(model, tokenizer, Xt, ctx.device, task), classes)
        # Save a compact FP16 state without mutating the training model/optimizer.
        half_state = {k: v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu()
                      for k, v in model.state_dict().items()}
        model.save_pretrained(current, state_dict=half_state, safe_serialization=True)
        tokenizer.save_pretrained(current)
        atomic_json(current / "classes.json", classes)
        ctx.publish(frame, score, {"algorithm": "transformer", "feature_mode": "transformer", "epoch": epoch + 1,
                                  "model_name": model_name, "split_hash": split_hash, "refit_all_rows": False},
                    artifacts={p.name: p for p in current.iterdir() if p.is_file()})
        best = score
    if best is None:
        raise RuntimeError("No transformer checkpoint completed within budget")


def predict_neural_checkpoint(cfg, task, candidate, device):
    import torch
    if task["kind"] == "timeseries":
        checkpoint = torch.load(candidate / "model.pt", map_location="cpu", weights_only=True)
        X, ids, _ = sequence_tensor(read_csv(cfg, task["test_csv"], task), task)
        X = (X - checkpoint["mean"].numpy()) / checkpoint["std"].numpy()
        classes = checkpoint["classes"]
        model = sensor_net(len(task["sensor_columns"]), len(classes))
        model.load_state_dict(checkpoint["state_dict"])
        model.to(device)
        probs = tensor_probs(model, X, device, int(task.get("batch_size", 128)))
    else:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        tokenizer = AutoTokenizer.from_pretrained(candidate, local_files_only=True)
        model = AutoModelForSequenceClassification.from_pretrained(candidate, local_files_only=True).float().to(device)
        test = read_csv(cfg, task["test_csv"], task)
        X = test[task["text_columns"]].fillna("").astype(str).agg(" ".join, axis=1)
        ids = test[task["id_column"]]
        classes = json.loads((candidate / "classes.json").read_text())
        probs = text_probs(model, tokenizer, X, device, task)
    return output_probs(cfg, task, ids, probs, classes)

```

### Python 소스: race90/tabular_gpu.py

```python
"""GPU CatBoost candidates with native checkpoints that also infer on CPU."""
import json
import time
import numpy as np
from sklearn.preprocessing import LabelEncoder
from .data import load_dataset
from .io import atomic_json, make_submission
from .models import split_indices, metric_score, predictions, MINIMIZE


def prepare_features(frame, schema):
    frame = frame.copy()
    frame.columns = frame.columns.map(str)
    frame = frame[schema["columns"]].copy()
    for column in frame:
        if column in schema["categorical"]:
            frame[column] = frame[column].astype(object).where(frame[column].notna(), "__missing__").astype(str)
        else:
            frame[column] = frame[column].astype(float).replace([np.inf, -np.inf], np.nan)
    return frame


class CatBoostPredictor:
    def __init__(self, model, classes=None, threads=1):
        self.model, self.threads = model, threads
        if classes is not None:
            self.classes_ = np.asarray(classes)

    def predict(self, X):
        result = np.asarray(self.model.predict(X, task_type="CPU", thread_count=self.threads)).reshape(-1)
        return self.classes_[result.astype(int)] if hasattr(self, "classes_") else result

    def predict_proba(self, X):
        return self.model.predict_proba(X, task_type="CPU", thread_count=self.threads)


def train_tabular(ctx):
    from catboost import CatBoostClassifier, CatBoostRegressor
    X, Xt, y, ids, metadata = load_dataset(ctx.cfg, ctx.task)
    task = ctx.task
    a, b, split_hash = split_indices(task, y, metadata, ctx.cfg["seed"])
    classification = task["objective"] == "classification"
    encoder = LabelEncoder().fit(y) if classification else None
    labels = encoder.transform(y) if classification else y.to_numpy()
    schema = {"columns": [str(c) for c in X.columns],
              "categorical": [str(c) for c in X.columns if c not in X.select_dtypes(include="number").columns],
              "classes": encoder.classes_.tolist() if classification else None}
    X, Xt = prepare_features(X, schema), prepare_features(Xt, schema)
    schema_path, checkpoint = ctx.output / "schema-current.json", ctx.output / "catboost-current.cbm"
    atomic_json(schema_path, schema)
    model_class = CatBoostClassifier if classification else CatBoostRegressor
    kwargs = dict(task_type="GPU" if ctx.device.startswith("cuda") else "CPU",
                  thread_count=ctx.threads, random_seed=ctx.cfg["seed"],
                  depth=int(task.get("depth", 6)), learning_rate=float(task.get("learning_rate", .06)),
                  cat_features=schema["categorical"], verbose=False, allow_writing_files=False)
    if ctx.device.startswith("cuda"):
        kwargs["devices"] = "0"  # CUDA_VISIBLE_DEVICES already isolates the assigned GPU.
    best, best_model, best_seconds = None, None, 0
    for iterations in task.get("catboost_iterations", [100, 300, 800]):
        if best_model is not None and ctx.seconds_left() < best_seconds * iterations / best_model.tree_count_ * 1.5 + 10:
            break
        model = model_class(iterations=int(iterations), **kwargs)
        started = time.monotonic()
        model.fit(X.iloc[a], labels[a], eval_set=(X.iloc[b], labels[b]),
                  early_stopping_rounds=int(task.get("patience", 40)), use_best_model=True)
        elapsed = time.monotonic() - started
        predictor = CatBoostPredictor(model, schema["classes"], ctx.threads)
        score = metric_score(predictor, X.iloc[b], y.iloc[b], task)
        improved = best is None or (score < best if task["metric"] in MINIMIZE else score > best)
        if not improved:
            continue
        model.save_model(checkpoint)
        frame = make_submission(ctx.cfg, task, ids, predictions(predictor, Xt, task))
        details = {"algorithm": model_class.__name__, "feature_mode": "catboost", "split_hash": split_hash,
                   "train_rows": len(a), "valid_rows": len(b), "iterations": model.tree_count_, "refit_all_rows": False}
        ctx.publish(frame, score, details, artifacts={"model.cbm": checkpoint, "schema.json": schema_path})
        best, best_model, best_seconds = score, model, elapsed
    if best_model is None:
        raise RuntimeError("No CatBoost checkpoint completed")
    if task.get("refit", True) and ctx.seconds_left() > best_seconds * len(y) / len(a) * 1.5 + 15:
        model = model_class(iterations=best_model.tree_count_, **kwargs)
        model.fit(X, labels)
        model.save_model(checkpoint)
        frame = make_submission(ctx.cfg, task, ids, predictions(CatBoostPredictor(model, schema["classes"], ctx.threads), Xt, task))
        ctx.publish(frame, best, {**details, "refit_all_rows": True},
                    artifacts={"model.cbm": checkpoint, "schema.json": schema_path})


def predict_tabular_checkpoint(cfg, task, candidate):
    from catboost import CatBoostClassifier, CatBoostRegressor
    schema = json.loads((candidate / "schema.json").read_text())
    _, Xt, _, ids, _ = load_dataset(cfg, task)
    model = CatBoostClassifier() if schema["classes"] is not None else CatBoostRegressor()
    model.load_model(candidate / "model.cbm")
    predictor = CatBoostPredictor(model, schema["classes"])
    return make_submission(cfg, task, ids, predictions(predictor, prepare_features(Xt, schema), task))

```

### Python 소스: race90/data.py

```python
import json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from .config import resolve_data
from .io import read_csv, atomic_json


def sensor_features(df, task):
    ident, sensors = task["id_column"], task["sensor_columns"]
    if task.get("time_column"):
        df = df.sort_values([ident, task["time_column"]], kind="stable")
    grouped = df.groupby(ident, sort=False)
    values = df[sensors].apply(pd.to_numeric, errors="raise").replace([np.inf, -np.inf], np.nan)
    # Per-recording features only: never cross sample boundaries or use labels.
    result = values.groupby(df[ident], sort=False).agg(["mean", "std", "min", "max", "median"])
    result.columns = [f"{sensor}_{stat}" for sensor, stat in result.columns]
    for sensor in sensors:
        g = values[sensor].groupby(df[ident], sort=False)
        result[f"{sensor}_rms"] = g.apply(lambda x: float(np.sqrt(np.nanmean(x.to_numpy() ** 2))))
        result[f"{sensor}_delta"] = g.last() - g.first()
        result[f"{sensor}_slope"] = g.apply(_slope)
    result["length"] = grouped.size()
    return result.reset_index()


def _slope(x):
    arr = x.to_numpy(dtype=float)
    valid = np.isfinite(arr)
    if valid.sum() < 2:
        return 0.0
    return float(np.polyfit(np.arange(len(arr))[valid], arr[valid], 1)[0])


def image_path(cfg, task, value):
    return resolve_data(cfg, str(Path(task.get("image_root", ".")) / str(value)))


def image_features(cfg, task, df):
    rows = []
    for filename in df[task["image_column"]]:
        with Image.open(image_path(cfg, task, filename)) as im:
            arr = np.asarray(im.convert("RGB").resize((16, 16)), dtype=np.float32) / 255
        hist = np.concatenate([np.histogram(arr[..., c], bins=16, range=(0, 1), density=False)[0] / 256
                               for c in range(3)])
        rows.append(np.concatenate([arr.mean(axis=(0, 1)), arr.std(axis=(0, 1)), hist,
                                    arr[::4, ::4].ravel()]))
    return pd.DataFrame(rows)


def load_dataset(cfg, task):
    train = read_csv(cfg, task["train_csv"], task)
    test = read_csv(cfg, task["test_csv"], task)
    for feature in task.get("dependency_features", []):
        parent = Path(cfg["dependency_outputs"][feature["task"]])
        key = feature.get("on", task["id_column"])
        train_extra = pd.read_csv(parent / feature["train_file"], dtype={key: train[key].dtype})
        test_extra = pd.read_csv(parent / feature["test_file"], dtype={key: test[key].dtype})
        if train_extra[key].isna().any() or test_extra[key].isna().any():
            raise ValueError("Dependency feature join keys must be non-null")
        if train_extra[key].duplicated().any() or test_extra[key].duplicated().any():
            raise ValueError("Dependency features require exactly one row per join key")
        if set(train[key]) != set(train_extra[key]) or set(test[key]) != set(test_extra[key]):
            raise ValueError("Dependency feature IDs don't cover training/test data")
        columns = set(train_extra) - {key}
        if columns != set(test_extra) - {key} or columns & (set(train) | set(test)):
            raise ValueError("Dependency train/test feature columns must match and not overwrite existing columns")
        train = train.merge(train_extra, on=key, how="left", validate="many_to_one", sort=False)
        test = test.merge(test_extra, on=key, how="left", validate="many_to_one", sort=False)
    target, ident = task["target"], task["id_column"]
    if target not in train or train[target].isna().any():
        raise ValueError("Target missing or contains nulls")
    if train[ident].isna().any() or test[ident].isna().any():
        raise ValueError("Train/test IDs must be non-null")
    if task["kind"] == "timeseries":
        if (train.groupby(ident)[target].nunique() > 1).any():
            raise ValueError("Each sensor recording must have exactly one target")
        group_column = task["split"].get("group_column")
        if group_column and (train.groupby(ident)[group_column].nunique(dropna=False) != 1).any():
            raise ValueError("Each sensor recording must belong to exactly one validation group")
        metadata = train.groupby(ident, sort=False).first().reset_index()
        metadata = recording_times(train, metadata, task)
        tr, te = sensor_features(train, task), sensor_features(test, task)
        metadata = metadata.set_index(ident).loc[tr[ident]].reset_index()
        y = metadata[target]
        X, Xt = tr.drop(columns=ident), te.drop(columns=ident)
        ids = te[ident]
        train = metadata
    elif task["kind"] == "text":
        columns = task["text_columns"]
        X = train[columns].fillna("").astype(str).agg(" ".join, axis=1)
        Xt = test[columns].fillna("").astype(str).agg(" ".join, axis=1)
        y, ids = train[target], test[ident]
    elif task["kind"] == "image":
        X, Xt = image_features(cfg, task, train), image_features(cfg, task, test)
        y, ids = train[target], test[ident]
    else:
        ignored = set(task.get("drop_columns", [])) | {target, ident}
        ignored |= {v for v in (task["split"].get("group_column"), task["split"].get("time_column")) if v}
        ignored |= {other["target"] for other in cfg.get("tasks", [])
                    if other.get("train_csv") == task["train_csv"] and other.get("target")}
        # Shared training files may contain the labels of several questions.
        # Infer only features available in test, excluding all declared targets.
        columns = task.get("feature_columns") or [c for c in train.columns if c not in ignored and c in test]
        if target in columns or ident in columns:
            raise ValueError("Target/ID columns cannot be features")
        if not columns:
            raise ValueError("No feature columns")
        X, Xt, y, ids = train[columns], test[columns], train[target], test[ident]
    if train[ident].duplicated().any() or ids.duplicated().any():
        raise ValueError("Duplicate sample ID (use timeseries kind for long sensor tables)")
    return X, Xt, y.reset_index(drop=True), ids.reset_index(drop=True), train


def recording_times(frame, metadata, task):
    if task["split"]["strategy"] == "time":
        column, ident = task["split"]["time_column"], task["id_column"]
        stamps = frame[column]
        if not pd.api.types.is_numeric_dtype(stamps):
            stamps = pd.to_datetime(stamps, errors="raise", utc=True)
        if stamps.isna().any():
            raise ValueError("Null validation timestamps")
        metadata[column] = metadata[ident].map(stamps.groupby(frame[ident], sort=False).min())
    return metadata


def audit(cfg, task, output, verify_images=True):
    report = {"task": task["name"], "kind": task["kind"], "warnings": []}
    tables = {}
    for key in ("train_csv", "test_csv", "sample_submission"):
        if task.get(key):
            df = read_csv(cfg, task[key], task)
            tables[key] = df
            report[key] = {"rows": len(df), "columns": list(df.columns),
                           "nulls": {k: int(v) for k, v in df.isna().sum().items()},
                           "dtypes": {k: str(v) for k, v in df.dtypes.items()}}
    ident = task["id_column"]
    sample, test = tables["sample_submission"], tables["test_csv"]
    if test[ident].isna().any() or (task["kind"] != "timeseries" and test[ident].duplicated().any()):
        raise ValueError("Invalid test IDs")
    if not set(task["prediction_columns"]) <= set(sample.columns):
        raise ValueError("Prediction columns missing from sample submission")
    if sample[ident].duplicated().any() or sample[ident].isna().any():
        raise ValueError("Invalid sample submission IDs")
    if set(sample[ident]) != set(test[ident]):
        raise ValueError("Sample/test ID mismatch")
    train = tables.get("train_csv")
    if train is not None:
        target = task.get("target")
        if train[ident].isna().any():
            raise ValueError("Null train IDs")
        if target and target not in train:
            raise ValueError("Target column missing from training data")
        if target and target in train:
            if train[target].isna().any():
                raise ValueError("Null targets")
            report["target_counts"] = {str(k): int(v) for k, v in train[target].value_counts().head(100).items()}
        if ident in train and set(train[ident]) & set(test[ident]):
            report["warnings"].append("Train/test IDs overlap; verify problem semantics")
        if task["kind"] != "timeseries" and train[ident].duplicated().any():
            raise ValueError("Duplicate train IDs")
        if task["kind"] == "timeseries" and (train.groupby(ident)[target].nunique() > 1).any():
            raise ValueError("Conflicting targets within recording")
        if task["split"]["strategy"] == "time":
            column = task["split"].get("time_column")
            if not column or column not in train or train[column].isna().any():
                raise ValueError("Time validation needs a complete time_column")
        if task["split"]["strategy"] == "group":
            column = task["split"].get("group_column")
            if not column or column not in train or train[column].isna().any():
                raise ValueError("Group validation needs a complete group_column")
            if task["kind"] == "timeseries" and (train.groupby(ident)[column].nunique() != 1).any():
                raise ValueError("Each sensor recording must belong to exactly one validation group")
    if verify_images and task["kind"] in {"image", "yolo"}:
        image_tables = [test] + ([train] if train is not None else [])
        checked = 0
        for df in image_tables:
            for value in df[task["image_column"]]:
                with Image.open(image_path(cfg, task, value)) as im:
                    im.verify()
                checked += 1
        report["verified_images"] = checked
    if task["kind"] == "yolo":
        from .vision import normalized_yolo_yaml
        _, counts = normalized_yolo_yaml(cfg, task, output)
        report["yolo_split_images"] = counts
    atomic_json(Path(output) / "audit.json", report)
    return report

```

### Python 소스: custom_adapters/example.py

```python
"""Copy this adapter and change it to match the actual problem statement."""
from pathlib import Path
import pandas as pd
from race90.config import resolve_data
from race90.io import make_submission, read_csv


def solve(ctx):
    """CPU baseline or GPU upgrade; ctx.stage tells you which phase is running."""
    # Parent directories contain the immutable chosen model and published artifacts.
    for name, directory in ctx.dependencies.items():
        print(f"Dependency {name}: {directory}", flush=True)
    test = read_csv(ctx.cfg, ctx.task["test_csv"], ctx.task)
    # Replace with actual validation, training and inference. The external runner
    # enforces a hard deadline, but polling ctx.seconds_left() permits clean saves.
    raise NotImplementedError("Implement this problem's modeling and CSV schema before running")
    # frame = make_submission(ctx.cfg, ctx.task, test[ctx.task["id_column"]], predictions)
    # ctx.publish(frame, validation_score, {"algorithm": "my-model", "feature_mode": "custom"},
    #             artifacts={"train_features.csv": train_oof_path,
    #                        "test_features.csv": test_features_path,
    #                        "model.pt": checkpoint_path})


def export_submission(cfg, task, test, results):
    """YOLO exporter: return a DataFrame exactly matching sample_submission."""
    raise NotImplementedError("Implement the exact detection output format required by the question")

```

### Python 소스: custom_adapters/competition.py

```python
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC


def radar_features(data):
    rows = []
    for sid, g in data.groupby('sample_id', sort=True):
        g = g.sort_values('t_ms')
        f = {'sample_id': sid, 'count': len(g), 'times': g.t_ms.nunique(),
             'sensors': g.sensor_id.nunique()}
        cols = ['rcs_dbsm', 'radial_velocity_comp_mps', 'radial_velocity_mps',
                'range_m', 'azimuth_rad', 'x_m', 'y_m']
        for c in cols:
            v = g[c].to_numpy()
            for name, value in [('mean', v.mean()), ('std', v.std()), ('min',v.min()),
                                ('max',v.max()), ('q10',np.quantile(v,.1)),
                                ('median',np.median(v)), ('q90',np.quantile(v,.9))]:
                f[c+'_'+name] = value
        t = g.t_ms.to_numpy()/1000
        tc = t-t.mean()
        denom = np.dot(tc,tc)+1e-12
        residuals=[]
        for c in ['x_m','y_m','rcs_dbsm','radial_velocity_comp_mps']:
            v=g[c].to_numpy(); slope=np.dot(tc,v-v.mean())/denom
            residual=v-v.mean()-slope*tc
            f[c+'_slope']=slope; f[c+'_resid_std']=residual.std()
            f[c+'_resid_qspan']=np.quantile(residual,.9)-np.quantile(residual,.1)
            if c in ['x_m','y_m']: residuals.append(residual)
        eig=np.linalg.eigvalsh(np.cov(residuals))
        f['spatial_eigen_small'],f['spatial_eigen_large']=eig
        f['track_speed']=np.hypot(f['x_m_slope'],f['y_m_slope'])
        f['comp_abs_mean']=g.radial_velocity_comp_mps.abs().mean()
        counts=g.groupby('t_ms').size().to_numpy()
        f['detections_per_time_mean']=counts.mean();f['detections_per_time_std']=counts.std()
        for c in ['x_m','y_m','rcs_dbsm','radial_velocity_comp_mps']:
            s=g.groupby('t_ms')[c].std().fillna(0)
            f[c+'_within_time_std']=s.mean()
        for sensor in [1,2,3,4]:
            sg=g[g.sensor_id==sensor]
            f['sensor_'+str(sensor)+'_fraction']=len(sg)/len(g)
        rows.append(f)
    return pd.DataFrame(rows).set_index('sample_id').fillna(0)


def text_model(c=3.0, ngrams=(2,5)):
    return Pipeline([('features',FeatureUnion([
        ('char',TfidfVectorizer(analyzer='char',ngram_range=ngrams,sublinear_tf=True)),
        ('word',TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True))])),
        ('model',LinearSVC(C=c, max_iter=5000, random_state=42))])


def solve(ctx):
    root=Path(ctx.cfg['data_root'])
    train=pd.read_csv(root/ctx.task['train_csv'])
    test=pd.read_csv(root/ctx.task['test_csv'])
    radar=ctx.task['name']=='q1'
    if radar:
        X=radar_features(train); Xt=radar_features(test)
        y=train.groupby('sample_id').label.first().reindex(X.index).to_numpy()
        ids=Xt.index.to_numpy()
    else:
        X=train.report; Xt=test.report; y=train.fault_type.to_numpy();ids=test.id.to_numpy()
    folds=list(StratifiedKFold(5,shuffle=True,random_state=42).split(np.zeros(len(y)),y))
    options=([('extra_trees',{})] if ctx.stage=='baseline' else
             [('catboost_d4',{'depth':4}),('catboost_d6',{'depth':6}),('catboost_d3',{'depth':3})]) if radar else [('text_svc',{'c':3.0})]
    for name,params in options:
        if ctx.seconds_left()<20: break
        def make_model():
            if name=='extra_trees':
                return ExtraTreesClassifier(n_estimators=400,max_features=.8,n_jobs=ctx.threads,random_state=42)
            if name.startswith('catboost'):
                from catboost import CatBoostClassifier
                return CatBoostClassifier(iterations=1200,learning_rate=.035,loss_function='MultiClass',
                    task_type='GPU' if ctx.device!='cpu' else 'CPU',devices='0',
                    thread_count=ctx.threads,verbose=False,random_seed=42,l2_leaf_reg=5,**params)
            return text_model(**params)
        oof=np.empty(len(y),dtype=object)
        for fold,(a,b) in enumerate(folds):
            m=make_model();m.fit(X.iloc[a],y[a]);oof[b]=np.asarray(m.predict(X.iloc[b])).ravel()
            print(name,'fold',fold,accuracy_score(y[b],oof[b]),flush=True)
        score=accuracy_score(y,oof) if radar else f1_score(y,oof,average='macro')
        m=make_model();m.fit(X,y);pred=np.asarray(m.predict(Xt)).ravel()
        frame=pd.DataFrame({ctx.task['id_column']:ids,ctx.task['target']:pred})
        oofpath=ctx.output/(name+'_oof.csv')
        pd.DataFrame({'truth':y,'prediction':oof}).to_csv(oofpath,index=False)
        artifacts={'oof.csv':oofpath}
        if name.startswith('catboost'):
            modelpath=ctx.output/(name+'.cbm');m.save_model(str(modelpath));artifacts['model.cbm']=modelpath
        ctx.publish(frame,float(score),{'algorithm':name,'validation':'5-fold stratified OOF; sample-level for radar',
                     'refit_all_rows':True,'features':list(X.columns) if radar else 'char + word TF-IDF'},
                    model=None if name.startswith('catboost') else m,artifacts=artifacts)

```

### Python 소스: custom_adapters/radar_improved.py

```python
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from custom_adapters.competition import radar_features


def sensor_geometry(data):
    transforms={}
    for sid,g in data.groupby('sensor_id'):
        a=np.c_[g.range_m*np.cos(g.azimuth_rad),g.range_m*np.sin(g.azimuth_rad),np.ones(len(g))]
        transforms[int(sid)]=np.linalg.lstsq(a,g[['x_m','y_m']].values,rcond=None)[0]
    return transforms


def features(data,geometry):
    basic=radar_features(data);rows=[]
    for sid,g in data.groupby('sample_id',sort=True):
        g=g.sort_values('t_ms');n=len(g);f={'sample_id':sid}
        t=g.t_ms.to_numpy()/1000;t=t-t.mean()
        pos=g[['x_m','y_m']].to_numpy();pos=pos-pos.mean(0)
        v=g.radial_velocity_comp_mps.to_numpy();r=g.rcs_dbsm.to_numpy()
        directions=[]
        for a,sen in zip(g.azimuth_rad,g.sensor_id):
            rot=geometry[int(sen)][:2,:]
            directions.append(np.array([np.cos(a),np.sin(a)])@rot)
        u=np.array(directions)
        velocity=np.linalg.lstsq(u,v,rcond=None)[0]
        f['doppler_speed']=np.linalg.norm(velocity)
        f['doppler_vx'],f['doppler_vy']=velocity
        f['doppler_resid_std']=np.std(v-u@velocity)
        f['doppler_resid_mad']=np.median(np.abs(v-u@velocity-np.median(v-u@velocity)))
        f['direction_condition']=np.linalg.cond(u.T@u)
        for ridge in [.01,.1,1,10]:
            vel=np.linalg.solve(u.T@u+ridge*np.eye(2),u.T@v)
            f['speed_ridge_'+str(ridge)]=np.linalg.norm(vel)
        # Joint least squares: positions at different times and radial velocities.
        for w in [.1,1,10]:
            vel=np.linalg.solve(u.T@u+w*np.dot(t,t)*np.eye(2),u.T@v+w*(t[:,None]*pos).sum(0))
            f['speed_joint_'+str(w)]=np.linalg.norm(vel)
        temporal_vel=(t[:,None]*pos).sum(0)/(np.dot(t,t)+1e-12)
        for mode,vel in [('raw',np.zeros(2)),('time',temporal_vel),('doppler',velocity)]:
            p=pos-t[:,None]*vel
            cov=p.T@p/max(n-1,1);e,rot=np.linalg.eigh(cov);pca=p@rot
            for i in range(2):
                f[mode+'_eig'+str(i)]=e[i]
                f[mode+'_pca_skew'+str(i)]=skew(pca[:,i])
                f[mode+'_pca_kurt'+str(i)]=kurtosis(pca[:,i])
                for q in [0.8,0.9,0.95,1.0]:
                    f[mode+'_span'+str(i)+'_'+str(q)]=np.quantile(pca[:,i],q)-np.quantile(pca[:,i],1-q)
            f[mode+'_xy_cov']=cov[0,1]
            f[mode+'_area']=np.sqrt(max(np.linalg.det(cov),0))
            f[mode+'_ratio']=e[1]/max(e[0],1e-8)
            f[mode+'_xvar']=cov[0,0];f[mode+'_yvar']=cov[1,1]
        within_cov=np.zeros((2,2));dof=0
        for _,gg in g.groupby('t_ms'):
            pp=gg[['x_m','y_m']].to_numpy();pp=pp-pp.mean(0)
            within_cov+=pp.T@pp;dof+=max(len(gg)-1,0)
        within_cov/=max(dof,1);e=np.linalg.eigvalsh(within_cov)
        f['within_eig0'],f['within_eig1']=e
        f['within_xvar']=within_cov[0,0];f['within_yvar']=within_cov[1,1]
        f['within_dof']=dof
        for name,val in [('rcs',r),('velocity',v),('absvelocity',np.abs(v))]:
            f[name+'_skew']=skew(val);f[name+'_kurt']=kurtosis(val)
            f[name+'_iqr']=np.quantile(val,.75)-np.quantile(val,.25)
            f[name+'_mad']=np.median(np.abs(val-np.median(val)))
        rows.append(f)
    extra=pd.DataFrame(rows).set_index('sample_id')
    return pd.concat([basic,extra],axis=1).replace([np.inf,-np.inf],np.nan).fillna(0)


if __name__=='__main__':
    root=Path('competition/q1');train=pd.read_csv(root/'data/train.csv');test=pd.read_csv(root/'data/test.csv')
    geo=sensor_geometry(train)
    X=features(train,geo);Xt=features(test,geo)
    X.to_csv(root/'features_improved.csv');Xt.to_csv(root/'features_test_improved.csv')
    train.groupby('sample_id').label.first().reindex(X.index).to_csv(root/'labels.csv')
    import json
    (root/'sensor_geometry.json').write_text(json.dumps({k:v.tolist() for k,v in geo.items()}))
    print(X.shape,flush=True)

```

### Python 소스: custom_adapters/__init__.py

```python
"""Competition-specific adapters live here and are bundled into Notebooks."""

```

### Python 소스: custom_adapters/text_improved.py

```python
import re
import numpy as np
import pandas as pd


def core_report(text):
    # Main observation follows one of the report's structural headers.
    headers = [r'운용자가\s*', r'정비반 보고:\s*', r'기능 시험 결과\s*',
               r'다음 사항을 확인함\.\s*',r'현장 정비 기록: .*?에서\s*',
               r'\[현상\]\s*',r'\[점검 메모\] .*?에서\s*',
               r'정비 메모 — .*?:\s*',r'운용자 인계(?:사항| 기록)\([^)]*\):\s*',
               r'(?:에 대한 )?재현 시험(?:에서| 결과,)\s*',
               r'점검 결과,\s*',r'운용 후 보고:\s*']
    found = []
    for pat in headers:
        m=re.search(pat,text)
        if m:found.append(m)
    if found:
        m=min(found,key=lambda x:x.start());text=text[m.end():]
    # A normal finding may be inserted immediately after the header.
    parts=text.split('.')
    while len(parts)>1 and re.search(r'가능성은 낮으며|특이사항이 없었다|정상으로 확인됐다|뒷받침하는 징후는 발견되지 않았다',parts[0]):
        parts=parts[1:]
    text='.'.join(parts).strip()
    endings = [r'이라고 보고함',r'15분 점검 중',r'정비 후 첫 운용에서',
               r'외기 온도는 정상 범위였음',r'증상은 2회 연속 재현됨',
               r'주변 환경 조건은 정상 범위였다',r'동일 현상이 재시험에서도',
               r'짧은 확인 운전에서도',r'동일 조건에서 두 차례',
               r'정지 후 다시 시작하면',r'\[대상\]',r'\[기타\]']
    text=re.split('|'.join(endings),text)[0]
    text=text.split('.')[0]
    return text.strip(' .')


if __name__=='__main__':
    from pathlib import Path
    from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
    from sklearn.metrics import f1_score
    from custom_adapters.competition import text_model
    root=Path('competition/q2')
    train=pd.read_csv(root/'data/train.csv');test=pd.read_csv(root/'data/test.csv')
    train['core']=train.report.map(core_report);test['core']=test.report.map(core_report)
    train.to_csv(root/'train_core.csv',index=False);test.to_csv(root/'test_core.csv',index=False)
    print('Train cores',train.core.nunique(),'test cores',test.core.nunique())
    splits=list(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(train.core,train.fault_type,train.core))
    m=text_model(c=1.0,ngrams=(1,4))
    p=cross_val_predict(m,train.core,train.fault_type,cv=splits)
    print('GroupCV',f1_score(train.fault_type,p,average='macro'))
    m.fit(train.core,train.fault_type)
    scores=m.decision_function(test.core);test['pred']=m.classes_[scores.argmax(1)]
    test['margin']=np.sort(scores,axis=1)[:,-1]-np.sort(scores,axis=1)[:,-2]
    test.to_csv(root/'core_review.csv',index=False)
    for _,r in test.drop_duplicates('core').iterrows():print(r.id,r.pred,round(r.margin,2),r.core)

```

In [1]:
from pathlib import Path
import base64, io, json, os, sys, tempfile, zipfile

# 온라인 Notebook에서 실제 데이터가 있는 경로로 필요하면 수정하세요.
DATA_ROOT = Path(os.environ.get("RACE90_DATA_ROOT", str(Path.cwd()))).resolve()
RECOMPUTE = False
import importlib.util, subprocess
missing = [package for module, package in {"numpy": "numpy", "pandas": "pandas", "yaml": "PyYAML"}.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
WORK = Path(tempfile.mkdtemp(prefix="race90-notebook-"))
SOURCE_BUNDLE = "..."
ARTIFACT_BUNDLE = "UEsDBBQAAAAIAFk5Sl3q/c85nAMAAHobAAAOAAAAc3VibWlzc2lvbi5jc3Z1mM1qXEkUg/fzLL0oqf6XYWjsgImDJ+MwqyYhWQx4IJD3hzgbz6L0eVldPn3Vks7RPT+//Pfj5fvt32+Xly9fv7/88fzuVn7/6fLp8+Pt8/31+nB9ejt1PK2Xh/d3959uz9f7938+XN/O2+X++u75n+O8xyoDqszLx+vTX48fPlwfbndPj39/fPtkxTqb7qv8/8nbmfK3yvnZBVjVQu0OdwGpEKlWqL7zE7pQFeusYoezihVafnIDUo/EkCfcjnwacFbEWYWfAKu1nr9CDYzWHs6AzToTmrrgCVC1DXG26NFmvF/j/Yb3o0/bwPszY2sL/2Pn364j5q78Hd1QCfzaQck9MNwRcUfH9qjlDnhHyagGoB2xDw/AOpDh0aE+aHqAc0dEOwDtRHYnOneCc2dw7kS8s+MnA+ojw3MBOkC9gOOFmFfozQs4XqDnFfS8AOua4S6gXIByA8oNSt7YrXZgdiOzO+DccfpsZHQD1r1DHZUST3OeUMn6VaFZK0hPKtmzKtSlhAlKJSNWyTNXIUFJaQpJuR8L8pMEaJVmkIRYdWpYovkj7fP2a36KT+iI06kTCxOUDDh96lfOXVgOGA1MOvtUNftU9cyIqmcfUk1ZQpX8qQqqrSPURr1CelINPLag1Ua9Vi3y+JqY4ve13G3Vcg5WAyZbYLIBk5CV1EGvPTDZqduqo2IhKakDo5iW1HN6UEd/duhDgxKEIDFpQD8aiBtTk0bwKiQmjcBwTEsaccpgVtKESTOR44lY5/muoxk77wx+nQEjpCNNes/RivN0BRUv4HKdiUGvuShVBe0uYHFhR4J8pIU4d8S543TBdKSQjrRhtmxKvdo5CWqnd1ftcwuhfXZelzxbXKj7GtKRS5oyLqdaXU5PupxadTm16kL9xyV50iEFWYgNNkmGJGQlxVpZscYkZNwmGfOQRaq1A+aYiBw2SnbkEbZJhm2SHfh03jkY8pAdGcVdknGX5Jq7kGvODK6R10redM3edM1T1BVZDenIIR255dnpBgpuoGBIR477JDfA2VC/Lc1Pt4Cyp15r2CEZ05Fhi+ROGcGQjozpyLhLMuYjQz4y5iOP6NkRPDtAxyN03xGZHcDsAM+Oc74YdkieMGNwg2TYIBlTkSfoGLdInsjuBNfOgBkzkmNG8oqcLlRzyElegdUFfRj2R14pNXhFt67g1p3fZAy7I+/0zuaQjIx7I++o3B0mzQ7JISQix21RLacjfwFQSwMEFAAAAAgAWTlKXYp8j3PJAgAAQwoAAA0AAABtYW5pZmVzdC5qc29ujVbBbqMwEL33K1DObQQJkLB732s/YFVZxh5StwYT22lhq/772kBScOyml0ieeTPvzRDZ7+MuilYaq9fVr2h1TFb39qw0PoANlFgBZw1MYSKkDcfrfVoMEYIbyijWgBid4x/iLIW4SKu0HEsptNBQaAgDZYAfn0O0Bi0ZsYWYkJPEpD+jNWZ8AJqjCWB+EJLp59piodMSIy3BtLof82+YWxlMNBaQPVSC00gZmGYVAxo9Pv75HSlctxweOLwBjyohI4kplucWEiqmEeYcSfFuqbU8wZSrAOuTHJT/HSJ2dHFq9FRsV8jqixy7KmiUkLOAJArRUtWoBtx4wkpTH5j5sDXuPNFjEnvpKPMSHos5HBsUR2YxgjDdIyLqFtWtctWGYEv1wWbsJ72W04VQy2nD8p3pg+2+28btRdzcwc3xb05+c+ifzOsZtTkAuvpXTlFnpAnKPEhH+hh05J6ZXIkTeC4L/2P1ST8jo9+RNs8s5C1KWKBiIXOeWEhdsi/lLormkrurLXbuBjt3e527uc7dWufZWOduq7+i7l3q3qXuXerepe491L1LPczIRQvLkLkrGb0efgwfVXulddmi97foQy2+btFln0vc08zJuR2DN53DEML5GL/HOgrMSVs4sAM0SNXmZQolOZaHmSbz7JFXpFqAGflAhkv3IjMvLRD7birUgkT2FfsB4urDvpu3mTWeZP9d8vIJwojQyoIV48OLElSZLQyGwE1twqltOJV+pYbMk/kdTQyRxh0ARVgbe5DsiiSPiziL11maFPvNbnROz3iT5dab5AUmBS1Jnu3jOCflDmLYFFWGkyLfFyXsKmOFyHafpGkKOYVNGe+KIk+3yW5b7cvtaJAmi7KN47H9qayZUkYcorpvYeacRtMz2TPjiM52h+MS+Dl2mQVLY5jMmF9OZ1ULCnz9IkrOynOxENWaqDdb93T3+R9QSwECFAMUAAAACABZOUpd6v3POZwDAAB6GwAADgAAAAAAAAAAAAAApIEAAAAAc3VibWlzc2lvbi5jc3ZQSwECFAMUAAAACABZOUpdinyPc8kCAABDCgAADQAAAAAAAAAAAAAApIHIAwAAbWFuaWZlc3QuanNvblBLBQYAAAAAAgACAHcAAAC8BgAAAAA="
for payload, dest in [(SOURCE_BUNDLE, WORK / "source"), (ARTIFACT_BUNDLE, WORK / "model")]:
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
        for name in archive.namelist():
            if not (dest / name).resolve().is_relative_to(dest.resolve()):
                raise ValueError("Unsafe bundled path")
        archive.extractall(dest)
sys.path.insert(0, str(WORK / "source"))
for name in list(sys.modules):
    if name == "race90" or name.startswith("race90."):
        del sys.modules[name]
CFG = json.loads("{\"seed\": 42, \"tasks\": [{\"name\": \"q1\", \"kind\": \"custom\", \"adapter\": \"custom_adapters.competition:solve\", \"resource\": \"gpu\", \"train_csv\": \"q1/data/train.csv\", \"test_csv\": \"q1/data/test.csv\", \"sample_submission\": \"q1/sample_submission.csv\", \"notebook\": \"q1/\ub808\uc774\ub354 \uc2dc\uacc4\uc5f4 \ub370\uc774\ud130 \uae30\ubc18 \uc774\ub3d9 \ud45c\uc801 \uc2dd\ubcc4\ud558\uae30.ipynb\", \"id_column\": \"sample_id\", \"target\": \"label\", \"prediction_columns\": [\"label\"], \"metric\": \"accuracy\", \"baseline_seconds\": 600, \"upgrade_seconds\": 1500, \"improve\": true, \"depends_on\": [], \"objective\": \"classification\", \"prediction_type\": \"label\", \"split\": {\"strategy\": \"stratified\", \"fraction\": 0.2}}], \"data_root\": \".\", \"project_root\": \".\", \"runpod\": {\"enabled\": false}, \"total_minutes\": 90, \"dependency_outputs\": {}}")
CFG["data_root"] = str(DATA_ROOT)
CFG["dependency_outputs"] = {name: str(WORK / "model" / path) for name, path in CFG["dependency_outputs"].items()}
TASK = CFG["tasks"][0]
print("문항:", TASK["name"], "데이터:", DATA_ROOT)


문항: q1 데이터: /config/workspace/race90/competition


In [2]:
# 실제 학습 로직은 위 소스의 train_cpu/train_resnet/train_yolo에 포함되어 있습니다.
# 원하면 CPU 기본 모델 학습을 이 Notebook 안에서 다시 실행할 수 있습니다.
RETRAIN_CPU = False
if RETRAIN_CPU:
    import time
    from race90.process import run_command
    from race90.io import atomic_json
    deadline = time.time() + 600
    payload = WORK / "retrain.json"
    atomic_json(payload, {"cfg": CFG, "task": TASK, "output": str(WORK / "retrained"),
                          "stage": "baseline", "deadline": deadline, "threads": 1, "device": "cpu"})
    env = os.environ.copy()
    env["PYTHONPATH"] = str(WORK / "source") + os.pathsep + env.get("PYTHONPATH", "")
    for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
        env[key] = "1"
    try:
        run_command([sys.executable, "-m", "race90", "_worker", str(payload)], deadline, env=env)
    except (TimeoutError, RuntimeError) as error:
        print("완료된 후보로 복구합니다:", error)
    from race90.selection import select_best
    selected, _ = select_best(CFG, TASK, WORK / "retrained")
    MODEL_DIR = selected
else:
    MODEL_DIR = WORK / "model"


In [3]:
import pandas as pd
from race90.io import read_csv, read_saved_submission, validate_submission, save_csv, sha256
if RECOMPUTE:
    from race90.models import predict_checkpoint
    submission = predict_checkpoint(CFG, TASK, MODEL_DIR, device="cpu")
    print("저장 모델로 CPU 추론을 다시 실행했습니다.")
else:
    # 이 예측은 하네스의 학습/추론 단계에서 실제 생성된 결과입니다.
    template = read_csv(CFG, TASK["sample_submission"], TASK)
    manifest = json.loads((MODEL_DIR / "manifest.json").read_text())
    submission = read_saved_submission(MODEL_DIR / "submission.csv", template, TASK, manifest)
    print("학습 단계에서 저장된 예측을 복원했습니다.")
template = read_csv(CFG, TASK["sample_submission"], TASK)
validate_submission(submission, template, TASK)
FINAL_CSV = Path(os.environ.get("RACE90_OUTPUT_DIR", str(Path.cwd()))) / (TASK["name"] + "_submission.csv")
save_csv(FINAL_CSV, submission)
model_report = json.loads((MODEL_DIR / "manifest.json").read_text())
print("검증 결과:", json.dumps(model_report, ensure_ascii=False, indent=2))
print("제출 파일:", FINAL_CSV, "행:", len(submission), "SHA256:", sha256(FINAL_CSV))
display(submission.head())


학습 단계에서 저장된 예측을 복원했습니다.
검증 결과: {
  "task": "q1",
  "stage": "baseline",
  "score": 0.849,
  "candidate_id": "baseline-054e094f4b",
  "dependencies": {},
  "metric": "accuracy",
  "details": {
    "algorithm": "extra_trees",
    "validation": "5-fold stratified OOF; sample-level for radar",
    "refit_all_rows": true,
    "features": [
      "count",
      "times",
      "sensors",
      "rcs_dbsm_mean",
      "rcs_dbsm_std",
      "rcs_dbsm_min",
      "rcs_dbsm_max",
      "rcs_dbsm_q10",
      "rcs_dbsm_median",
      "rcs_dbsm_q90",
      "radial_velocity_comp_mps_mean",
      "radial_velocity_comp_mps_std",
      "radial_velocity_comp_mps_min",
      "radial_velocity_comp_mps_max",
      "radial_velocity_comp_mps_q10",
      "radial_velocity_comp_mps_median",
      "radial_velocity_comp_mps_q90",
      "radial_velocity_mps_mean",
      "radial_velocity_mps_std",
      "radial_velocity_mps_min",
      "radial_velocity_mps_max",
      "radial_velocity_mps_q10",
      "radial_velocity

,sample_id,label
0,VA_000001,TWO_WHEELER
1,VA_000002,TWO_WHEELER
2,VA_000003,LIGHT_VEHICLE
3,VA_000004,HEAVY_VEHICLE
4,VA_000005,TWO_WHEELER
